# 01. Project Overview ? Few-Shot EfficientNet-B2 & Pseudo-Labeling

**Kompetisi**: DSC 2026 Waste Classification  
**Task**: TASK 08 ? Unified Pipeline: Few-Shot Learning + Pseudo-Labeling + Kaggle Submission  
**Backbone Arsitektur**: `EfficientNet-B2` (pretrained `ImageNet-1K_V1`)  
**Strategi Semi-Supervised**: Teacher-Student Pseudo-Labeling  

### Latar Belakang & Batasan Regulasi
Dalam skenario kompetisi ini, kita menghadapi tantangan klasifikasi citra dengan **anggaran pelabelan manual yang sangat terbatas (Few-Shot)**:
- **Maksimal 5 citra per kelas** atau total **maksimal 40 citra manual** untuk 8 kelas target.
- Sisa dataset training (~1.876 citra) diperlakukan sebagai **Unlabeled Pool** tanpa akses ke ground-truth label asli.
- Data test resmi kompetisi (480 citra pada `data/competition/test/`) hanya digunakan untuk inferensi akhir (submission), **tanpa kebocoran data (no leakage)**.

### Alur Eksekusi Eksperimen (Unified Single Notebook):
1. **Bagian A ? Persiapan (01-11)**: Environment, audit regulasi, partisi labeled (40 citra) & unlabeled (1.876 citra), audit anti-kebocoran, dan protokol 5-Fold Stratified Cross-Validation.
2. **Bagian B ? Preprocessing (12-17)**: Preprocessing resmi EfficientNet-B2 (288x288, Bicubic, ImageNet Normalization) dan augmentasi data terkendali.
3. **Bagian C ? Few-Shot Teacher (18-26)**: Pelatihan model Teacher pada 40 citra berlabel, evaluasi Out-Of-Fold CV, dan checkpointing.
4. **Bagian D ? Pseudo-Labeling (27-33)**: Inferensi unlabeled pool menggunakan Teacher, analisis confidence scores, penyaringan pseudo-label (threshold 0.95), dan analisis keseimbangan kelas.
5. **Bagian E ? Student Training (34-41)**: Pelatihan model Student menggunakan data gabungan (40 labeled + pseudo-labels) dengan pembobotan loss semi-supervised.
6. **Bagian F ? Evaluasi (42-47)**: Perbandingan performa Teacher vs Student, metrik validasi, laporan klasifikasi, matriks konfusi, dan analisis overfitting/underfitting.
7. **Bagian G ? Kaggle Submission (48-54)**: Audit data test kompetisi, setup inferensi mandiri (standalone), prediksi 480 citra test, pembuatan CSV submission, dan verifikasi integritas CSV.

*Seluruh proses dikendalikan melalui flag eksekusi manual (default False). Pengguna menjalankan eksekusi secara mandiri.*

In [1]:
# ==============================================================================
# SECTION 01: PROJECT METADATA & ARCHITECTURE OVERVIEW
# ==============================================================================
PIPELINE_METADATA = {
    "project": "DSC_2026_Waste_Classification",
    "task": "TASK_08_FewShot_PseudoLabeling_EfficientNet_B2",
    "model_backbone": "EfficientNet-B2 (torchvision IMAGENET1K_V1)",
    "num_classes": 8,
    "classes": ["cardboard", "electronic", "foodwaste", "glass", "metal", "paper", "plastic", "textile"],
    "max_labels_per_class": 5,
    "max_total_labels": 40,
    "cv_folds": 5,
    "confidence_threshold_default": 0.95,
    "pseudo_loss_weight": 0.5,
    "input_resolution": (288, 288),
    "random_seed": 42
}

print("=" * 75)
print(f"PROYEK          : {PIPELINE_METADATA['project']}")
print(f"TASK            : {PIPELINE_METADATA['task']}")
print(f"BACKBONE        : {PIPELINE_METADATA['model_backbone']}")
print(f"RESOLUSI INPUT  : {PIPELINE_METADATA['input_resolution']}")
print(f"ANGGARAN LABEL  : Maksimal {PIPELINE_METADATA['max_labels_per_class']} citra/kelas ({PIPELINE_METADATA['max_total_labels']} citra total)")
print(f"KELAS TARGET    : {', '.join(PIPELINE_METADATA['classes'])}")
print(f"RANDOM SEED     : {PIPELINE_METADATA['random_seed']}")
print("=" * 75)

PROYEK          : DSC_2026_Waste_Classification
TASK            : TASK_08_FewShot_PseudoLabeling_EfficientNet_B2
BACKBONE        : EfficientNet-B2 (torchvision IMAGENET1K_V1)
RESOLUSI INPUT  : (288, 288)
ANGGARAN LABEL  : Maksimal 5 citra/kelas (40 citra total)
KELAS TARGET    : cardboard, electronic, foodwaste, glass, metal, paper, plastic, textile
RANDOM SEED     : 42


# 02. Environment Setup & Manual Execution Flags

Bagian ini mengimpor pustaka standar dan saintifik, menetapkan random seed deterministik (42), mendeteksi perangkat keras akselerasi GPU (CUDA) dengan fallback CPU, dan mendeklarasikan seluruh **Manual Execution Flags**.

### Aturan Eksekusi Manual
Seluruh flag berat bernilai `False` secara default. Pengguna dapat mengaktifkan flag secara bertahap saat ingin menjalankan tahapan tertentu di lingkungan lokal:
- `RULES_VERIFIED`: Konfirmasi manual atas buku aturan lomba.
- `RUN_FEWSHOT_PREPARATION`: Membagi labeled pool (40) dan unlabeled pool (1.876).
- `RUN_CROSS_VALIDATION`: Menjalankan 5-Fold Cross-Validation teacher.
- `RUN_TEACHER_TRAINING`: Melatih Final Teacher pada 40 citra berlabel.
- `RUN_PSEUDO_LABEL_GENERATION`: Menghasilkan raw pseudo-label pada data unlabeled.
- `RUN_PSEUDO_LABEL_SELECTION`: Menyaring pseudo-label dengan threshold confidence.
- `RUN_STUDENT_TRAINING`: Melatih model Student pada data gabungan.
- `RUN_TEACHER_STUDENT_EVALUATION`: Menjalankan evaluasi komparatif Teacher vs Student.
- `RUN_TEST_INFERENCE`: Memprediksi 480 citra test kompetisi.
- `RUN_SUBMISSION_GENERATION`: Membuat file submission CSV resmi.

In [2]:
# ==============================================================================
# SECTION 02: ENVIRONMENT SETUP, DEPENDENCIES & MANUAL FLAGS
# ==============================================================================
import os
import sys
import json
import time
import math
import random
import shutil
import hashlib
import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Sequence, Set
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import sklearn.metrics as metrics
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights
import torchvision.transforms.functional as TF

try:
    from torchvision.transforms import v2
    USE_TRANSFORMS_V2 = True
except ImportError:
    import torchvision.transforms as v2
    USE_TRANSFORMS_V2 = False

from tqdm.auto import tqdm

# --- REPRODUCIBILITY SEED SETUP ---
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# --- HARDWARE ACCELERATION DETECTION ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

print("STATUS ENVIRONMENT:")
print(f"  Python Version      : {sys.version.split()[0]}")
print(f"  PyTorch Version     : {torch.__version__}")
print(f"  Torchvision Version : {torchvision.__version__}")
print(f"  Device              : {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU Device Name     : {torch.cuda.get_device_name(0)}")
    print(f"  GPU VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print(f"  Mixed Precision     : {'Enabled (AMP)' if USE_AMP else 'Disabled (FP32)'}")
print(f"  Transforms Engine   : {'Torchvision v2' if USE_TRANSFORMS_V2 else 'Torchvision v1'}")

# ==============================================================================
# MANUAL EXECUTION FLAGS (ALL DEFAULT FALSE)
# Ubah flag ke True secara manual sebelum menjalankan cell tahapan terkait
# ==============================================================================
RULES_VERIFIED = False
RUN_FEWSHOT_PREPARATION = False
RUN_CROSS_VALIDATION = False
RUN_TEACHER_TRAINING = False
RUN_PSEUDO_LABEL_GENERATION = False
RUN_PSEUDO_LABEL_SELECTION = False
RUN_STUDENT_TRAINING = False
RUN_TEACHER_STUDENT_EVALUATION = False
RUN_TEST_INFERENCE = False
RUN_SUBMISSION_GENERATION = False

print()
print("STATUS MANUAL EXECUTION FLAGS:")
print(f"  RULES_VERIFIED                 = {RULES_VERIFIED}")
print(f"  RUN_FEWSHOT_PREPARATION        = {RUN_FEWSHOT_PREPARATION}")
print(f"  RUN_CROSS_VALIDATION           = {RUN_CROSS_VALIDATION}")
print(f"  RUN_TEACHER_TRAINING           = {RUN_TEACHER_TRAINING}")
print(f"  RUN_PSEUDO_LABEL_GENERATION    = {RUN_PSEUDO_LABEL_GENERATION}")
print(f"  RUN_PSEUDO_LABEL_SELECTION     = {RUN_PSEUDO_LABEL_SELECTION}")
print(f"  RUN_STUDENT_TRAINING           = {RUN_STUDENT_TRAINING}")
print(f"  RUN_TEACHER_STUDENT_EVALUATION = {RUN_TEACHER_STUDENT_EVALUATION}")
print(f"  RUN_TEST_INFERENCE             = {RUN_TEST_INFERENCE}")
print(f"  RUN_SUBMISSION_GENERATION      = {RUN_SUBMISSION_GENERATION}")

STATUS ENVIRONMENT:
  Python Version      : 3.11.9
  PyTorch Version     : 2.10.0+cu128
  Torchvision Version : 0.25.0+cu128
  Device              : cuda
  GPU Device Name     : NVIDIA GeForce RTX 3050 Laptop GPU
  GPU VRAM Total      : 4.00 GB
  Mixed Precision     : Enabled (AMP)
  Transforms Engine   : Torchvision v2

STATUS MANUAL EXECUTION FLAGS:
  RULES_VERIFIED                 = False
  RUN_FEWSHOT_PREPARATION        = False
  RUN_CROSS_VALIDATION           = False
  RUN_TEACHER_TRAINING           = False
  RUN_PSEUDO_LABEL_GENERATION    = False
  RUN_PSEUDO_LABEL_SELECTION     = False
  RUN_STUDENT_TRAINING           = False
  RUN_TEACHER_STUDENT_EVALUATION = False
  RUN_TEST_INFERENCE             = False
  RUN_SUBMISSION_GENERATION      = False


# 03. Official Rules Verification & Approval Gate

Sebelum memulai pelatihan, integritas kepatuhan regulasi lomba wajib diaudit secara teliti:
1. **Identifikasi Dokumen Resmi**:
   Handbook resmi kompetisi telah tersedia di direktori `docs/` (`HANDBOOK PRELIMINARY ROUND DSC GELAR RASA 2026  (1).pdf`).
2. **Ringkasan Ketentuan Regulasi Resmi (Berdasarkan Halaman 7?8 Handbook)**:
   - **Batas Hand Labeling Data Train**: Peserta tidak disarankan melakukan hand labeling pada data train. Jika melakukan hand labeling, dibatasi **maksimal 5 gambar per kelas** (total maksimal 40 gambar untuk 8 kelas target dari total 1.920 data train).
   - **Integritas Data Test**: Peserta **dilarang keras** melakukan hand labeling terhadap data test (480 citra) dan **dilarang menggunakan data test** dalam proses pelatihan model.
   - **Pretrained Model**: Peserta **diperbolehkan** menggunakan pretrained model selama model tersebut dijalankan oleh peserta dalam lingkungan pengerjaan lokal/kompetisi.
   - **Larangan Dataset Eksternal**: Peserta **hanya diperbolehkan** menggunakan dataset yang disediakan panitia dan **dilarang menggunakan dataset eksternal** dalam pengerjaan proyek.
   - **Larangan AutoML, LLM, VLM, Generative AI**: Peserta **dilarang keras** menggunakan AutoML/LLM, VLM, generative AI, atau layanan AI pihak ketiga untuk membuat model, melakukan klasifikasi, atau menghasilkan prediksi terhadap data kompetisi. Library/framework pendukung yang tidak melakukan inference terhadap data kompetisi tetap diperbolehkan.
   - **Status Pseudo-Labeling**: Teknik *pseudo-labeling* atau *semi-supervised learning* **tidak disebutkan secara eksplisit** dalam handbook. Namun, karena model dilatih secara mandiri pada data kompetisi internal tanpa dataset eksternal dan tetap menghormati kuota 5 hand label per kelas, teknik ini digunakan sebagai strategi teknis internal.
   - **Kewajiban Satu Notebook Reproducible**: Peserta diwajibkan mengumpulkan **hanya satu file notebook (.ipynb)** yang mencakup seluruh tahapan, bersifat *reproducible* sehingga dapat dijalankan ulang oleh panitia dan menghasilkan skor yang sama dengan file submission CSV.
3. **Approval Gate Independen**:
   Meskipun dokumen handbook resmi telah berhasil dideteksi dan diaudit, flag `RULES_VERIFIED` **tetap bernilai `False` secara default**. Pengguna sendiri yang harus menyetujui dan mengubah flag menjadi `True` secara manual di Section 02 setelah memvalidasi interpretasi aturan tersebut.

In [3]:
# ==============================================================================
# SECTION 03: OFFICIAL RULES VERIFICATION & APPROVAL GATE
# ==============================================================================
def resolve_project_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent
    ]
    for candidate in candidates:
        if (candidate / 'data' / 'competition').exists() or (candidate / 'data' / 'extracted').exists():
            return candidate.resolve()
    return Path.cwd().resolve()

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = resolve_project_root()

def detect_competition_handbook(docs_dir: Path) -> List[Dict[str, Any]]:
    # Mencari berkas PDF handbook/regulasi secara case-insensitive pada direktori docs
    matched_files = []
    if not docs_dir.exists():
        return matched_files
        
    keywords = ['handbook', 'preliminary', 'dsc', 'gelar', 'rasa', 'rule', 'regulasi', 'panduan', 'guide', 'ketentuan', 'pedoman']
    
    for item in sorted(docs_dir.iterdir(), key=lambda p: p.name.lower()):
        if item.is_file() and item.suffix.lower() == '.pdf':
            name_lower = item.name.lower()
            if any(kw in name_lower for kw in keywords):
                matched_files.append({
                    'name': item.name,
                    'path': item.resolve(),
                    'relative_path': item.relative_to(PROJECT_ROOT).as_posix(),
                    'size_bytes': item.stat().st_size
                })
    return matched_files

DOCS_DIR = PROJECT_ROOT / 'docs'
detected_handbooks = detect_competition_handbook(DOCS_DIR)

print('=' * 75)
print('HASIL AUDIT DETEKSI DOKUMEN REGULASI RESMI:')
print(f'Direktori Pencarian   : {DOCS_DIR.resolve()}')
print(f'Dokumen Ditemukan     : {len(detected_handbooks) > 0}')

if detected_handbooks:
    for idx, doc_info in enumerate(detected_handbooks, 1):
        print(f'\n[DOKUMEN {idx}]')
        print(f'  Nama Berkas   : {doc_info["name"]}')
        print(f'  Path Lengkap  : {doc_info["path"]}')
        print(f'  Ukuran Berkas : {doc_info["size_bytes"] / (1024*1024):.2f} MB')
else:
    print('  [PERINGATAN] Berkas PDF regulasi/handbook tidak ditemukan di docs/.')

# Ringkasan Eksekutif Aturan Resmi Berdasarkan Dokumen (Handbook Halaman 7-8)
OFFICIAL_RULES_SUMMARY = {
    'hand_labeling_train_limit': 'Maksimal 5 gambar per kelas (total 40 gambar manual dari 1920 data train).',
    'test_data_restriction': 'Dilarang hand labeling pada data test dan dilarang menggunakan data test untuk training.',
    'pretrained_model_allowed': 'Pretrained model diperbolehkan selama dijalankan dalam lingkungan lokal peserta.',
    'external_dataset_prohibited': 'Dilarang menggunakan dataset eksternal; hanya boleh memakai dataset panitia.',
    'automl_llm_genai_prohibited': 'Dilarang memakai AutoML/LLM, VLM, Generative AI, atau API pihak ketiga untuk model/prediksi.',
    'pseudo_labeling_status': 'Tidak dijelaskan secara eksplisit di handbook (digunakan sebagai strategi teknis mandiri).',
    'single_notebook_reproducible': 'Wajib 1 file notebook (.ipynb) yang reproducible dan menghasilkan skor sama dengan CSV submission.',
    'max_submissions_per_day': 3,
    'submission_deadline': '14 November 2026 pukul 23.59 WIB'
}

print('\n' + '=' * 75)
print('RINGKASAN ATURAN KOMPETISI RESMI (HANDBOOK HALAMAN 7-8):')
print(f'1. Kuota Hand Labeling Data Train : {OFFICIAL_RULES_SUMMARY["hand_labeling_train_limit"]}')
print(f'2. Larangan Data Test             : {OFFICIAL_RULES_SUMMARY["test_data_restriction"]}')
print(f'3. Izin Pretrained Model          : {OFFICIAL_RULES_SUMMARY["pretrained_model_allowed"]}')
print(f'4. Larangan Dataset Eksternal     : {OFFICIAL_RULES_SUMMARY["external_dataset_prohibited"]}')
print(f'5. Larangan AutoML/LLM/GenAI      : {OFFICIAL_RULES_SUMMARY["automl_llm_genai_prohibited"]}')
print(f'6. Status Pseudo-Labeling         : {OFFICIAL_RULES_SUMMARY["pseudo_labeling_status"]}')
print(f'7. Kewajiban Single Notebook      : {OFFICIAL_RULES_SUMMARY["single_notebook_reproducible"]}')
print(f'8. Batas Pengumpulan Submission   : {OFFICIAL_RULES_SUMMARY["submission_deadline"]}')
print('=' * 75)

# Status Manual Approval Gate (Tidak diubah otomatis menjadi True)
print(f'\nStatus Approval Gate: RULES_VERIFIED = {RULES_VERIFIED}')
if not RULES_VERIFIED:
    print('[APPROVAL GATE: PENDING] Dokumen regulasi telah dideteksi dan diringkas,')
    print('namun status verifikasi tetap \'False\' sesuai prinsip manual execution control.')
    print('Silakan tinjau ringkasan di atas, lalu ubah flag RULES_VERIFIED = True')
    print('pada Section 02 secara manual untuk menyetujui kepatuhan aturan.')
else:
    print('[APPROVAL GATE: VERIFIED] Status kepatuhan telah disetujui pengguna.')

HASIL AUDIT DETEKSI DOKUMEN REGULASI RESMI:
Direktori Pencarian   : D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste_Classification\docs
Dokumen Ditemukan     : True

[DOKUMEN 1]
  Nama Berkas   : HANDBOOK PRELIMINARY ROUND DSC GELAR RASA 2026  (1).pdf
  Path Lengkap  : D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste_Classification\docs\HANDBOOK PRELIMINARY ROUND DSC GELAR RASA 2026  (1).pdf
  Ukuran Berkas : 5.46 MB

RINGKASAN ATURAN KOMPETISI RESMI (HANDBOOK HALAMAN 7-8):
1. Kuota Hand Labeling Data Train : Maksimal 5 gambar per kelas (total 40 gambar manual dari 1920 data train).
2. Larangan Data Test             : Dilarang hand labeling pada data test dan dilarang menggunakan data test untuk training.
3. Izin Pretrained Model          : Pretrained model diperbolehkan selama dijalankan dalam lingkungan lokal peserta.
4. Larangan Dataset Eksternal     : Dilarang menggunakan dataset eksternal; hanya boleh memakai dataset panitia.
5. Larangan AutoML/LLM/GenAI      : Dilarang memakai AutoM

# 04. Dataset Path Configuration (Portable & Self-Contained)

Notebook ini dirancang sepenuhnya portabel dan bebas dari hardcoded drive Windows (`D:\...`).
Fungsi `resolve_project_root()` akan mendeteksi direktori utama proyek secara dinamis, sehingga notebook ini dapat dieksekusi oleh juri atau panitia kompetisi di sistem operasi apa pun (Linux, macOS, Windows).

In [4]:
# ==============================================================================
# SECTION 04: DATASET PATH CONFIGURATION
# ==============================================================================
def resolve_project_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent
    ]
    for candidate in candidates:
        if (candidate / "data" / "competition").exists() or (candidate / "data" / "extracted").exists():
            return candidate.resolve()
    return Path.cwd().resolve()

PROJECT_ROOT = resolve_project_root()

# Direktori Sumber Data Asli
DATA_RAW_TRAIN_DIR = PROJECT_ROOT / "data" / "extracted" / "v1" / "train"
DATA_TEST_DIR = PROJECT_ROOT / "data" / "competition" / "test"
SAMPLE_SUBMISSION_PATH = PROJECT_ROOT / "data" / "competition" / "SampleSubmission_DSC.csv"

# Direktori Manifest & Artefak TASK 08
MANIFEST_DIR = PROJECT_ROOT / "data" / "manifests" / "fewshot_pseudolabel_b2_v1"
CHECKPOINT_DIR = PROJECT_ROOT / "models" / "checkpoints" / "fewshot_pseudolabel_b2_v1"
CV_CHECKPOINT_DIR = CHECKPOINT_DIR / "cv"
TEACHER_CHECKPOINT_DIR = CHECKPOINT_DIR / "teacher"
STUDENT_CHECKPOINT_DIR = CHECKPOINT_DIR / "student"

OUTPUT_METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
OUTPUT_LOGS_DIR = PROJECT_ROOT / "outputs" / "logs"
OUTPUT_FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
SUBMISSION_DIR = PROJECT_ROOT / "Final Dataset for Submission to Kaggle"
SUBMISSION_CSV_PATH = SUBMISSION_DIR / "submission_fewshot_pseudolabel_b2.csv"

# Buat seluruh direktori artefak jika belum ada
for folder in [
    MANIFEST_DIR, CV_CHECKPOINT_DIR, TEACHER_CHECKPOINT_DIR, STUDENT_CHECKPOINT_DIR,
    OUTPUT_METRICS_DIR, OUTPUT_LOGS_DIR, OUTPUT_FIGURES_DIR, SUBMISSION_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT           : {PROJECT_ROOT}")
print(f"DATA_RAW_TRAIN_DIR     : {DATA_RAW_TRAIN_DIR} (Exists: {DATA_RAW_TRAIN_DIR.exists()})")
print(f"DATA_TEST_DIR          : {DATA_TEST_DIR} (Exists: {DATA_TEST_DIR.exists()})")
print(f"SAMPLE_SUBMISSION_PATH : {SAMPLE_SUBMISSION_PATH} (Exists: {SAMPLE_SUBMISSION_PATH.exists()})")
print(f"MANIFEST_DIR           : {MANIFEST_DIR}")
print(f"CHECKPOINT_DIR         : {CHECKPOINT_DIR}")
print(f"SUBMISSION_CSV_PATH    : {SUBMISSION_CSV_PATH}")


PROJECT_ROOT           : D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste_Classification
DATA_RAW_TRAIN_DIR     : D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste_Classification\data\extracted\v1\train (Exists: True)
DATA_TEST_DIR          : D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste_Classification\data\competition\test (Exists: True)
SAMPLE_SUBMISSION_PATH : D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste_Classification\data\competition\SampleSubmission_DSC.csv (Exists: True)
MANIFEST_DIR           : D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste_Classification\data\manifests\fewshot_pseudolabel_b2_v1
CHECKPOINT_DIR         : D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste_Classification\models\checkpoints\fewshot_pseudolabel_b2_v1
SUBMISSION_CSV_PATH    : D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste_Classification\Final Dataset for Submission to Kaggle\submission_fewshot_pseudolabel_b2.csv


# 05. Dataset Integrity Audit

Sebelum melakukan partisi dan pemrosesan, kita melakukan audit integritas fisik terhadap seluruh berkas citra pada disk:
1. Memastikan file citra dapat dibuka secara utuh oleh pustaka `PIL.Image`.
2. Mendeteksi citra rusak, korup, atau tidak terbaca (0 unreadable files diharapkan).
3. Memeriksa format ekstensi dan mode warna (RGB).

In [5]:
# ==============================================================================
# SECTION 05: DATASET INTEGRITY AUDIT
# ==============================================================================
def audit_image_files(directory: Path, allowed_exts: Tuple[str, ...] = (".jpg", ".jpeg", ".png")) -> Dict[str, Any]:
    if not directory.exists():
        return {"exists": False, "total_files": 0, "readable": 0, "corrupted": 0, "issues": []}
    
    image_paths = [p for p in directory.rglob("*") if p.is_file() and p.suffix.lower() in allowed_exts]
    readable_count = 0
    corrupted_count = 0
    issues = []
    
    for path in image_paths:
        try:
            with Image.open(path) as img:
                img.verify()
            readable_count += 1
        except Exception as err:
            corrupted_count += 1
            issues.append((str(path), str(err)))
            
    return {
        "exists": True,
        "directory": str(directory),
        "total_files": len(image_paths),
        "readable": readable_count,
        "corrupted": corrupted_count,
        "issues": issues
    }

print("Menjalankan audit integritas berkas citra...")
train_audit = audit_image_files(DATA_RAW_TRAIN_DIR)
test_audit = audit_image_files(DATA_TEST_DIR)

print(f"RAW TRAIN DIR : {train_audit['total_files']} files ditemukan | {train_audit['readable']} readable | {train_audit['corrupted']} corrupted")
print(f"COMP TEST DIR : {test_audit['total_files']} files ditemukan | {test_audit['readable']} readable | {test_audit['corrupted']} corrupted")

if train_audit['corrupted'] > 0 or test_audit['corrupted'] > 0:
    print(f"[PERINGATAN] Ditemukan file korup: Train={train_audit['corrupted']}, Test={test_audit['corrupted']}")
else:
    print("[INTEGRITAS VALID] Seluruh berkas citra valid dan dapat dibaca dengan baik.")

Menjalankan audit integritas berkas citra...
RAW TRAIN DIR : 1916 files ditemukan | 1916 readable | 0 corrupted
COMP TEST DIR : 480 files ditemukan | 480 readable | 0 corrupted
[INTEGRITAS VALID] Seluruh berkas citra valid dan dapat dibaca dengan baik.


# 06. Class Definition & Mappings

Definisi delapan kelas resmi kompetisi `DSC_2026_Waste_Classification` dalam urutan abjad standar:
0: `cardboard`
1: `electronic`
2: `foodwaste`
3: `glass`
4: `metal`
5: `paper`
6: `plastic`
7: `textile`

In [6]:
# ==============================================================================
# SECTION 06: CLASS DEFINITION & MAPPINGS
# ==============================================================================
CLASS_NAMES = sorted([
    "cardboard",
    "electronic",
    "foodwaste",
    "glass",
    "metal",
    "paper",
    "plastic",
    "textile"
])
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for idx, name in enumerate(CLASS_NAMES)}

print(f"Jumlah Kelas : {NUM_CLASSES}")
print("Pemetaan Kelas Resmi:")
for idx, name in IDX_TO_CLASS.items():
    print(f"  [{idx}] {name}")

Jumlah Kelas : 8
Pemetaan Kelas Resmi:
  [0] cardboard
  [1] electronic
  [2] foodwaste
  [3] glass
  [4] metal
  [5] paper
  [6] plastic
  [7] textile


# 07. Labeling Budget Verification & Enforcement

Fungsi validator `verify_labeling_budget()` menjamin secara tegas bahwa:
1. Tidak ada kelas yang memiliki lebih dari **5 sampel berlabel**.
2. Total sampel berlabel manual tidak pernah melebihi **40 sampel**.
3. Jika kuota dilanggar, fungsi akan memunculkan `ValueError` seketika.

In [7]:
# ==============================================================================
# SECTION 07: LABELING BUDGET VERIFICATION
# ==============================================================================
def verify_labeling_budget(
    df: pd.DataFrame,
    class_col: str = "class_name",
    max_per_class: int = 5,
    max_total: int = 40
) -> bool:
    total_samples = len(df)
    if total_samples > max_total:
        raise ValueError(f"Pelanggaran Anggaran: Total sampel berlabel ({total_samples}) melebihi batas maksimum ({max_total})!")
    
    counts = df[class_col].value_counts()
    for cls_name, count in counts.items():
        if count > max_per_class:
            raise ValueError(f"Pelanggaran Anggaran: Kelas '{cls_name}' memiliki {count} sampel berlabel (maksimal {max_per_class})!")
            
    return True

# Test verifikasi pada dummy dataframe
dummy_ok = pd.DataFrame({"class_name": [c for c in CLASS_NAMES for _ in range(5)]})
assert verify_labeling_budget(dummy_ok, max_per_class=5, max_total=40) is True
print("[VALIDATOR TERUJI] Fungsi verify_labeling_budget berhasil memverifikasi kuota 40 label (5/kelas).")

[VALIDATOR TERUJI] Fungsi verify_labeling_budget berhasil memverifikasi kuota 40 label (5/kelas).


# 08. Few-Shot Labeled Sample Selection (5 Sampel per Kelas)

Prosedur seleksi sampel few-shot dari dataset training mentah (`DATA_RAW_TRAIN_DIR`):
- Seleksi deterministik berbasis seed (42) dengan pengurutan alfabetis stabil.
- Memilih tepat **5 sampel per kelas** (total 40 sampel).
- Menghitung nilai hash SHA-256 untuk audit integritas dan pelacakan permanen.
- Menyimpan manifest ke `data/manifests/fewshot_pseudolabel_b2_v1/labeled_pool_40.csv`.
- Dilindungi oleh flag `RUN_FEWSHOT_PREPARATION`. Jika manifest sudah ada di disk, manifest akan dimuat secara langsung.

In [ ]:
# ==============================================================================
# SECTION 08: FEW-SHOT LABELED SAMPLE SELECTION
# ==============================================================================
LABELED_POOL_PATH = MANIFEST_DIR / "labeled_pool_40.csv"

def compute_file_sha256(filepath: Path) -> str:
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

def select_fewshot_labeled_pool(
    raw_train_dir: Path,
    output_path: Path,
    samples_per_class: int = 5,
    seed: int = 42
) -> pd.DataFrame:
    records = []
    rng = random.Random(seed)
    
    excluded_noisy_basenames = {
        "1368_jpg.rf.1ca9d9e39e2ab975d0371ac6387ada31.jpg",
        "1548_jpg.rf.097efc120796cf28c589550dab63ec5a.jpg"
    }
    
    for cls_name in CLASS_NAMES:
        cls_dir = raw_train_dir / cls_name
        if not cls_dir.exists():
            raise FileNotFoundError(f"Direktori kelas tidak ditemukan: {cls_dir}")
            
        candidate_files = sorted([
            f for f in cls_dir.glob("*.*")
            if f.is_file() and f.suffix.lower() in [".jpg", ".jpeg", ".png"]
            and f.name not in excluded_noisy_basenames
        ])
        
        if len(candidate_files) < samples_per_class:
            raise ValueError(f"Kelas '{cls_name}' hanya memiliki {len(candidate_files)} citra (dibutuhkan {samples_per_class})!")
            
        selected_files = rng.sample(candidate_files, samples_per_class)
        selected_files = sorted(selected_files, key=lambda p: p.name)
        
        for file_path in selected_files:
            rel_path = file_path.relative_to(PROJECT_ROOT).as_posix()
            sha256_hash = compute_file_sha256(file_path)
            image_id = file_path.stem
            records.append({
                "image_id": image_id,
                "class_name": cls_name,
                "class_idx": CLASS_TO_IDX[cls_name],
                "relative_path": rel_path,
                "sha256": sha256_hash
            })
            
    df = pd.DataFrame(records)
    verify_labeling_budget(df, max_per_class=samples_per_class, max_total=samples_per_class * len(CLASS_NAMES))
    df.to_csv(output_path, index=False)
    return df

if LABELED_POOL_PATH.exists():
    labeled_pool_df = pd.read_csv(LABELED_POOL_PATH)
    verify_labeling_budget(labeled_pool_df)
    print(f"[LOADED] Manifest Labeled Pool dimuat dari disk: {len(labeled_pool_df)} sampel.")
elif RUN_FEWSHOT_PREPARATION:
    print("[RUNNING] Menjalankan seleksi Few-Shot Labeled Pool (40 sampel)...")
    labeled_pool_df = select_fewshot_labeled_pool(DATA_RAW_TRAIN_DIR, LABELED_POOL_PATH)
    print(f"[CREATED] Manifest Labeled Pool berhasil dibuat dan disimpan ke {LABELED_POOL_PATH}")
else:
    print("[STATUS] RUN_FEWSHOT_PREPARATION is False dan manifest belum tersedia.")
    print("         Ubah RUN_FEWSHOT_PREPARATION = True pada Section 02 untuk membuat labeled pool.")
    labeled_pool_df = pd.DataFrame(columns=["image_id", "class_name", "class_idx", "relative_path", "sha256"])

if len(labeled_pool_df) > 0:
    print(labeled_pool_df["class_name"].value_counts())

# 09. Unlabeled Pool Preparation (Anti-Leakage Guaranteed)

Sisa dataset training (~1.876 citra) di luar 40 labeled samples dialokasikan ke **Unlabeled Pool**.
**PERLINDUNGAN ANTI-LEAKAGE KETAT**:
- Manifest unlabeled pool **HANYA** menyimpan `image_id`, `relative_path`, dan `sha256`.
- Kolom `class_name` dan `class_idx` **DIHAPUS** secara permanen dari manifest ini.
- Hal ini menjamin bahwa DataLoader untuk data unlabeled tidak dapat mengakses true label asli.

In [ ]:
# ==============================================================================
# SECTION 09: UNLABELED POOL PREPARATION
# ==============================================================================
UNLABELED_POOL_PATH = MANIFEST_DIR / "unlabeled_pool.csv"

def build_unlabeled_pool(
    raw_train_dir: Path,
    labeled_df: pd.DataFrame,
    output_path: Path
) -> pd.DataFrame:
    labeled_sha256 = set(labeled_df["sha256"].tolist())
    records = []
    
    all_files = sorted([
        f for f in raw_train_dir.rglob("*.*")
        if f.is_file() and f.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ], key=lambda p: p.as_posix())
    
    for file_path in all_files:
        sha256_hash = compute_file_sha256(file_path)
        if sha256_hash in labeled_sha256:
            continue
            
        rel_path = file_path.relative_to(PROJECT_ROOT).as_posix()
        image_id = file_path.stem
        records.append({
            "image_id": image_id,
            "relative_path": rel_path,
            "sha256": sha256_hash
        })
        
    df = pd.DataFrame(records)
    assert "class_name" not in df.columns, "Bocoran: Kolom class_name ditemukan di unlabeled pool!"
    assert "class_idx" not in df.columns, "Bocoran: Kolom class_idx ditemukan di unlabeled pool!"
    
    df.to_csv(output_path, index=False)
    return df

if UNLABELED_POOL_PATH.exists():
    unlabeled_pool_df = pd.read_csv(UNLABELED_POOL_PATH)
    print(f"[LOADED] Manifest Unlabeled Pool dimuat dari disk: {len(unlabeled_pool_df)} sampel.")
elif RUN_FEWSHOT_PREPARATION and len(labeled_pool_df) > 0:
    print("[RUNNING] Mempersiapkan Unlabeled Pool (tanpa label)...")
    unlabeled_pool_df = build_unlabeled_pool(DATA_RAW_TRAIN_DIR, labeled_pool_df, UNLABELED_POOL_PATH)
    print(f"[CREATED] Manifest Unlabeled Pool berhasil dibuat: {len(unlabeled_pool_df)} citra.")
else:
    print("[STATUS] RUN_FEWSHOT_PREPARATION is False atau Labeled Pool belum tersedia.")
    unlabeled_pool_df = pd.DataFrame(columns=["image_id", "relative_path", "sha256"])

print(f"Kolom Manifest Unlabeled: {list(unlabeled_pool_df.columns)}")

# 10. Duplicate and Leakage Audit

Audit komprehensif untuk memastikan tidak ada kebocoran data (data leakage):
1. **Disjointness Labeled vs Unlabeled**: Tidak ada irisan hash SHA-256 antara Labeled Pool dan Unlabeled Pool.
2. **Disjointness Train vs Test**: Tidak ada irisan hash SHA-256 antara Train Pools (Labeled + Unlabeled) dan Competition Test Set (`data/competition/test/`).

In [ ]:
# ==============================================================================
# SECTION 10: DUPLICATE AND LEAKAGE AUDIT
# ==============================================================================
def audit_dataset_leakage(
    labeled_df: pd.DataFrame,
    unlabeled_df: pd.DataFrame,
    test_dir: Path
) -> Dict[str, Any]:
    labeled_hashes = set(labeled_df["sha256"]) if len(labeled_df) > 0 else set()
    unlabeled_hashes = set(unlabeled_df["sha256"]) if len(unlabeled_df) > 0 else set()
    
    test_hashes = set()
    if test_dir.exists():
        for test_file in test_dir.glob("*.*"):
            if test_file.is_file() and test_file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                test_hashes.add(compute_file_sha256(test_file))
                
    overlap_labeled_unlabeled = labeled_hashes & unlabeled_hashes
    overlap_labeled_test = labeled_hashes & test_hashes
    overlap_unlabeled_test = unlabeled_hashes & test_hashes
    
    return {
        "labeled_count": len(labeled_hashes),
        "unlabeled_count": len(unlabeled_hashes),
        "test_count": len(test_hashes),
        "overlap_labeled_unlabeled": len(overlap_labeled_unlabeled),
        "overlap_labeled_test": len(overlap_labeled_test),
        "overlap_unlabeled_test": len(overlap_unlabeled_test),
        "leakage_detected": (len(overlap_labeled_unlabeled) + len(overlap_labeled_test) + len(overlap_unlabeled_test)) > 0
    }

leakage_report = audit_dataset_leakage(labeled_pool_df, unlabeled_pool_df, DATA_TEST_DIR)
print("HASIL AUDIT ANTI-LEAKAGE:")
print(f"  Sampel Labeled Pool   : {leakage_report['labeled_count']}")
print(f"  Sampel Unlabeled Pool : {leakage_report['unlabeled_count']}")
print(f"  Sampel Test Kompetisi : {leakage_report['test_count']}")
print(f"  Overlap Labeled vs Unlabeled : {leakage_report['overlap_labeled_unlabeled']} (Diharapkan 0)")
print(f"  Overlap Labeled vs Test      : {leakage_report['overlap_labeled_test']} (Diharapkan 0)")
print(f"  Overlap Unlabeled vs Test    : {leakage_report['overlap_unlabeled_test']} (Diharapkan 0)")

if leakage_report['leakage_detected']:
    raise AssertionError("[FATAL LEAKAGE] Kebocoran data terdeteksi antar set!")
else:
    print("[LEAKAGE AUDIT PASS] Tidak ada tumpang tindih data. Integritas partisi terjamin 100%.")

# 11. Few-Shot Validation Protocol (5-Fold Stratified Cross-Validation)

Untuk mengevaluasi Teacher Model di bawah batasan 40 sampel manual, kita menggunakan **5-Fold Stratified Cross-Validation**:
- Setiap fold: **4 sampel/kelas untuk training (32 citra)** dan **1 sampel/kelas untuk validasi (8 citra)**.
- Seluruh 40 sampel dievaluasi secara out-of-fold tepat satu kali.
- Manifest fold disimpan ke `data/manifests/fewshot_pseudolabel_b2_v1/cv_folds.csv`.

In [ ]:
# ==============================================================================
# SECTION 11: FEW-SHOT VALIDATION PROTOCOL
# ==============================================================================
CV_FOLDS_PATH = MANIFEST_DIR / "cv_folds.csv"

def generate_cv_folds(labeled_df: pd.DataFrame, n_splits: int = 5, seed: int = 42) -> pd.DataFrame:
    df = labeled_df.copy().reset_index(drop=True)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    df["fold"] = -1
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df, df["class_idx"])):
        df.loc[val_idx, "fold"] = fold_idx
    return df

if CV_FOLDS_PATH.exists():
    cv_folds_df = pd.read_csv(CV_FOLDS_PATH)
    print(f"[LOADED] Manifest CV Folds dimuat dari disk ({len(cv_folds_df)} baris).")
elif RUN_FEWSHOT_PREPARATION and len(labeled_pool_df) > 0:
    cv_folds_df = generate_cv_folds(labeled_pool_df, n_splits=5, seed=42)
    cv_folds_df.to_csv(CV_FOLDS_PATH, index=False)
    print(f"[CREATED] Manifest CV Folds berhasil dibuat dan disimpan ke {CV_FOLDS_PATH}")
else:
    print("[STATUS] CV Folds belum dibuat (RUN_FEWSHOT_PREPARATION is False).")
    cv_folds_df = pd.DataFrame()

if len(cv_folds_df) > 0:
    for f in range(5):
        val_samples = cv_folds_df[cv_folds_df["fold"] == f]
        train_samples = cv_folds_df[cv_folds_df["fold"] != f]
        print(f"  Fold {f}: Train = {len(train_samples)} citra ({len(train_samples)//8}/kelas) | Val = {len(val_samples)} citra ({len(val_samples)//8}/kelas)")

# 12. Image Preprocessing (EfficientNet-B2 Pretrained Standard)

Sesuai bobot resmi `EfficientNet_B2_Weights.IMAGENET1K_V1`, pipeline preprocessing untuk validasi dan inferensi:
1. Konversi ke RGB.
2. Resize ke 288 (Bicubic Interpolation).
3. Center Crop 288x288.
4. Konversi ke Tensor float32 dalam skala [0.0, 1.0].
5. Normalisasi ImageNet standar:
   - Mean: `[0.485, 0.456, 0.406]`
   - Std: `[0.229, 0.224, 0.225]`

In [ ]:
# ==============================================================================
# SECTION 12: IMAGE PREPROCESSING
# ==============================================================================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 288

def get_eval_transforms(image_size: int = 288):
    if USE_TRANSFORMS_V2:
        return v2.Compose([
            v2.ToImage(),
            v2.Resize(image_size, interpolation=v2.InterpolationMode.BICUBIC, antialias=True),
            v2.CenterCrop(image_size),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ])
    else:
        return torchvision.transforms.Compose([
            torchvision.transforms.Resize(image_size, interpolation=torchvision.transforms.InterpolationMode.BICUBIC),
            torchvision.transforms.CenterCrop(image_size),
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ])

eval_transforms = get_eval_transforms(IMAGE_SIZE)
print("Transformasi Evaluasi & Inferensi:")
print(eval_transforms)

# 13. Training Augmentation (Controlled & Photorealistic)

Pada skenario data sedikit (Few-Shot), augmentasi data sangat penting untuk mencegah penghafalan (memorization), namun tidak boleh terlalu ekstrem agar tidak merusak representasi visual kelas:
- `RandomHorizontalFlip(p=0.5)`
- `RandomRotation(degrees=15)`
- `RandomAffine(degrees=0, shear=10)`
- `ColorJitter(brightness=0.10)`
- Output berukuran tepat 288x288 dengan Normalisasi ImageNet.

In [ ]:
# ==============================================================================
# SECTION 13: TRAINING AUGMENTATION
# ==============================================================================
def get_train_transforms(image_size: int = 288):
    if USE_TRANSFORMS_V2:
        return v2.Compose([
            v2.ToImage(),
            v2.Resize(image_size, interpolation=v2.InterpolationMode.BICUBIC, antialias=True),
            v2.CenterCrop(image_size),
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomRotation(degrees=15),
            v2.RandomAffine(degrees=0, shear=10),
            v2.ColorJitter(brightness=0.10),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ])
    else:
        return torchvision.transforms.Compose([
            torchvision.transforms.Resize(image_size, interpolation=torchvision.transforms.InterpolationMode.BICUBIC),
            torchvision.transforms.CenterCrop(image_size),
            torchvision.transforms.RandomHorizontalFlip(p=0.5),
            torchvision.transforms.RandomRotation(degrees=15),
            torchvision.transforms.RandomAffine(degrees=0, shear=10),
            torchvision.transforms.ColorJitter(brightness=0.10),
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ])

train_transforms = get_train_transforms(IMAGE_SIZE)
print("Transformasi Pelatihan (Augmentasi):")
print(train_transforms)

# 14. Visualization of Original Images

Visualisasi 1 contoh citra asli dari labeled pool untuk masing-masing dari 8 kelas sampah target.

In [ ]:
# ==============================================================================
# SECTION 14: VISUALIZATION OF ORIGINAL IMAGES
# ==============================================================================
def plot_sample_images_per_class(labeled_df: pd.DataFrame, project_root: Path):
    if len(labeled_df) == 0:
        print("[INFO] Labeled pool belum tersedia untuk divisualisasikan.")
        return
        
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    axes = axes.flatten()
    
    for idx, cls_name in enumerate(CLASS_NAMES):
        cls_rows = labeled_df[labeled_df["class_name"] == cls_name]
        if len(cls_rows) == 0:
            continue
        first_row = cls_rows.iloc[0]
        img_path = project_root / first_row["relative_path"]
        with Image.open(img_path) as img:
            axes[idx].imshow(img)
            axes[idx].set_title(f"[{first_row['class_idx']}] {cls_name} ({img.size[0]}x{img.size[1]})")
            axes[idx].axis("off")
            
    plt.suptitle("Contoh Citra Asli dari Labeled Pool (8 Kelas Target)", fontsize=14, weight="bold")
    plt.tight_layout()
    plt.show()

plot_sample_images_per_class(labeled_pool_df, PROJECT_ROOT)

# 15. Visualization of Augmented Images

Visualisasi variasi augmentasi pada satu citra sampel untuk memastikan transformasi tetap mempertahankan karakteristik semantik kelas objek.

In [ ]:
# ==============================================================================
# SECTION 15: VISUALIZATION OF AUGMENTED IMAGES
# ==============================================================================
def plot_augmented_variations(labeled_df: pd.DataFrame, project_root: Path):
    if len(labeled_df) == 0:
        print("[INFO] Labeled pool belum tersedia.")
        return
        
    sample_path = project_root / labeled_df.iloc[0]["relative_path"]
    with Image.open(sample_path) as raw_img:
        rgb_img = raw_img.convert("RGB")
        
    mean = np.array(IMAGENET_MEAN)
    std = np.array(IMAGENET_STD)
    
    fig, axes = plt.subplots(1, 5, figsize=(15, 3.5))
    axes[0].imshow(rgb_img)
    axes[0].set_title("Citra Asli")
    axes[0].axis("off")
    
    for i in range(1, 5):
        aug_tensor = train_transforms(rgb_img)
        np_img = aug_tensor.permute(1, 2, 0).cpu().numpy()
        np_img = std * np_img + mean
        np_img = np.clip(np_img, 0, 1)
        axes[i].imshow(np_img)
        axes[i].set_title(f"Augmentasi {i}")
        axes[i].axis("off")
        
    plt.suptitle(f"Variasi Augmentasi Training pada Kelas '{labeled_df.iloc[0]['class_name']}'", fontsize=12, weight="bold")
    plt.tight_layout()
    plt.show()

plot_augmented_variations(labeled_pool_df, PROJECT_ROOT)

# 16. Dataset and DataLoader Construction

Implementasi kelas PyTorch `Dataset` yang mandiri di dalam notebook:
- `FewShotLabeledDataset`: membaca citra berlabel, mengembalikan `(tensor, class_idx, image_id)`.
- `UnlabeledDataset`: membaca citra tanpa label, mengembalikan `(tensor, image_id)` ? **tanpa label asli**.
- `CompetitionTestDataset`: membaca citra test kompetisi untuk inferensi akhir.

In [ ]:
# ==============================================================================
# SECTION 16: DATASET AND DATALOADER CONSTRUCTION
# ==============================================================================
class FewShotLabeledDataset(Dataset):
    def __init__(self, df: pd.DataFrame, project_root: Path, transform: Optional[Callable] = None):
        self.df = df.reset_index(drop=True)
        self.project_root = project_root
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int, str]:
        row = self.df.iloc[idx]
        img_path = self.project_root / row["relative_path"]
        with Image.open(img_path) as img:
            image = img.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row["class_idx"]), str(row["image_id"])

class UnlabeledDataset(Dataset):
    def __init__(self, df: pd.DataFrame, project_root: Path, transform: Optional[Callable] = None):
        self.df = df.reset_index(drop=True)
        self.project_root = project_root
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, str]:
        row = self.df.iloc[idx]
        img_path = self.project_root / row["relative_path"]
        with Image.open(img_path) as img:
            image = img.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, str(row["image_id"])

class CompetitionTestDataset(Dataset):
    def __init__(self, image_paths: List[Path], transform: Optional[Callable] = None):
        self.image_paths = sorted(image_paths, key=lambda p: p.name)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, str]:
        path = self.image_paths[idx]
        with Image.open(path) as img:
            image = img.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, path.stem

print("[DATASET TERDEFINISI]")
print("  - FewShotLabeledDataset : (tensor, class_idx, image_id)")
print("  - UnlabeledDataset      : (tensor, image_id) [NO LABEL]")
print("  - CompetitionTestDataset: (tensor, image_id)")

# 17. Class Distribution Visualization

Visualisasi distribusi sampel berlabel (tepat 5 citra per kelas) dan proporsi data berlabel vs data tanpa label dalam pipeline eksperimen semi-supervised.

In [ ]:
# ==============================================================================
# SECTION 17: CLASS DISTRIBUTION VISUALIZATION
# ==============================================================================
def plot_dataset_proportions(labeled_df: pd.DataFrame, unlabeled_df: pd.DataFrame, test_dir: Path):
    labeled_count = len(labeled_df)
    unlabeled_count = len(unlabeled_df)
    test_count = len(list(test_dir.glob("*.*"))) if test_dir.exists() else 0
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    if labeled_count > 0:
        counts = labeled_df["class_name"].value_counts().reindex(CLASS_NAMES).fillna(0)
        bars = ax1.bar(counts.index, counts.values, color="steelblue", edgecolor="black")
        ax1.set_title(f"Distribusi Labeled Pool (Total: {labeled_count} Citra)", weight="bold")
        ax1.set_ylabel("Jumlah Sampel")
        ax1.tick_params(axis="x", rotation=45)
        for bar in bars:
            h = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., h + 0.1, f"{int(h)}", ha="center", va="bottom", weight="bold")
    else:
        ax1.text(0.5, 0.5, "Labeled pool belum dimuat", ha="center", va="center")
        
    sizes = [labeled_count, unlabeled_count, test_count]
    labels = [f"Labeled Manual ({labeled_count})", f"Unlabeled Pool ({unlabeled_count})", f"Comp Test ({test_count})"]
    colors = ["#2ca02c", "#1f77b4", "#ff7f0e"]
    
    if sum(sizes) > 0:
        ax2.pie(sizes, labels=labels, colors=colors, autopct="%1.1f%%", startangle=140, explode=(0.1, 0, 0))
        ax2.set_title("Proporsi Alokasi Citra dalam Proyek", weight="bold")
    else:
        ax2.text(0.5, 0.5, "Dataset belum dialokasikan", ha="center", va="center")
        
    plt.tight_layout()
    plt.show()

plot_dataset_proportions(labeled_pool_df, unlabeled_pool_df, DATA_TEST_DIR)

# 18. Teacher Model Initialization (EfficientNet-B2 Pretrained)

Inisialisasi model Teacher menggunakan arsitektur `EfficientNet-B2` dengan bobot resmi ImageNet-1K:
- Memuat bobot resmi `EfficientNet_B2_Weights.IMAGENET1K_V1`.
- Mempertahankan layer Dropout (0.3) bawaan torchvision.
- Mengganti layer proyeksi akhir linear dari 1408 fitur menjadi 8 kelas target.
- Membekukan seluruh layer fitur backbone (`features.parameters()`) agar pelatihan terfokus pada classifier head.

In [ ]:
# ==============================================================================
# SECTION 18: TEACHER MODEL INITIALIZATION
# ==============================================================================
def build_teacher_model(
    num_classes: int = 8,
    pretrained: bool = True,
    freeze_backbone: bool = True
) -> nn.Module:
    weights = EfficientNet_B2_Weights.IMAGENET1K_V1 if pretrained else None
    model = efficientnet_b2(weights=weights)
    
    in_features = model.classifier[1].in_features
    # Ganti classifier akhir menjadi 8 kelas
    model.classifier[1] = nn.Linear(in_features, num_classes)
    
    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False
            
    return model

teacher_model = build_teacher_model(num_classes=NUM_CLASSES, pretrained=True, freeze_backbone=True)
teacher_model = teacher_model.to(DEVICE)
print("[TEACHER INITIALIZED] EfficientNet-B2 Pretrained ImageNet-1K berhasil diinisialisasi.")

# 19. Model Architecture Summary

Inspeksi struktur parameter model:
- Total parameter: ~7,71 juta.
- Trainable parameter: 11.272 (hanya linear classifier head `1408 * 8 + 8 = 11.272`).
- Frozen parameter: ~7,70 juta.
- Uji forward pass sintetis dengan tensor shape `(2, 3, 288, 288)`.

In [ ]:
# ==============================================================================
# SECTION 19: MODEL ARCHITECTURE SUMMARY
# ==============================================================================
def inspect_model_parameters(model: nn.Module) -> Dict[str, Any]:
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params
    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "frozen_params": frozen_params
    }

param_summary = inspect_model_parameters(teacher_model)
print("PARAMETER SUMMARY:")
print(f"  Total Parameters     : {param_summary['total_params']:,}")
print(f"  Trainable Parameters : {param_summary['trainable_params']:,} (Head Only)")
print(f"  Frozen Parameters    : {param_summary['frozen_params']:,} (Backbone)")

# Forward pass sintetis untuk verifikasi output shape
with torch.no_grad():
    dummy_input = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
    dummy_output = teacher_model(dummy_input)
    print(f"  Input Shape          : {dummy_input.shape}")
    print(f"  Output Shape         : {dummy_output.shape} (Expected: (2, 8))")
    assert dummy_output.shape == (2, NUM_CLASSES)

# 20. Few-Shot Cross-Validation Configuration

Konfigurasi hyperparameter untuk 5-Fold Stratified Cross-Validation model Teacher:
- Optimizer: `AdamW` (learning rate = 1e-3, weight decay = 1e-4)
- Loss Function: `CrossEntropyLoss`
- Batch Size: 8
- Maksimum Epoch: 20 per fold
- Early Stopping: `patience = 5`, memantau `val_loss`
- LR Scheduler: `ReduceLROnPlateau` (factor = 0.5, patience = 3, min_lr = 1e-6)

In [ ]:
# ==============================================================================
# SECTION 20: FEW-SHOT CROSS-VALIDATION CONFIGURATION
# ==============================================================================
CV_CONFIG = {
    "experiment_id": "fewshot_pseudolabel_b2_cv",
    "num_folds": 5,
    "batch_size": 8,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "max_epochs": 20,
    "early_stopping_patience": 5,
    "scheduler_factor": 0.5,
    "scheduler_patience": 3,
    "min_lr": 1e-6
}

print("KONFIGURASI CROSS-VALIDATION:")
for k, v in CV_CONFIG.items():
    print(f"  {k:24}: {v}")

# 21. Manual Cross-Validation Execution (5 Folds)

Eksekusi pelatihan 5-Fold Cross-Validation untuk mengukur estimasi generalisasi out-of-fold yang jujur:
- Dilindungi oleh flag `RUN_CROSS_VALIDATION`.
- Monitoring per batch dengan `tqdm.auto`.
- Pencatatan metrik per epoch: train loss, train acc, val loss, val acc, macro precision, macro recall, macro F1.
- Checkpoint terbaik per fold disimpan ke `models/checkpoints/fewshot_pseudolabel_b2_v1/cv/fold_{i}/best.pt`.
- Riwayat lengkap disimpan secara persisten ke `outputs/metrics/fewshot_pseudolabel_b2_cv.json`.

In [ ]:
# ==============================================================================
# SECTION 21: MANUAL CROSS-VALIDATION EXECUTION
# ==============================================================================
CV_METRICS_PATH = OUTPUT_METRICS_DIR / "fewshot_pseudolabel_b2_cv.json"

def calculate_macro_metrics(y_true: List[int], y_pred: List[int], num_classes: int = 8) -> Dict[str, float]:
    acc = metrics.accuracy_score(y_true, y_pred)
    prec = metrics.precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = metrics.recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = metrics.f1_score(y_true, y_pred, average="macro", zero_division=0)
    return {"accuracy": float(acc), "macro_precision": float(prec), "macro_recall": float(rec), "macro_f1": float(f1)}

def run_cv_fold_training(fold_idx: int, cv_df: pd.DataFrame, config: Dict[str, Any]) -> Dict[str, Any]:
    print()
    print(f"--- MEMULAI FOLD {fold_idx} ---")
    train_sub_df = cv_df[cv_df["fold"] != fold_idx].copy()
    val_sub_df = cv_df[cv_df["fold"] == fold_idx].copy()
    
    train_ds = FewShotLabeledDataset(train_sub_df, PROJECT_ROOT, transform=train_transforms)
    val_ds = FewShotLabeledDataset(val_sub_df, PROJECT_ROOT, transform=eval_transforms)
    
    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False, num_workers=0)
    
    model = build_teacher_model(num_classes=NUM_CLASSES, pretrained=True, freeze_backbone=True).to(DEVICE)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=config["scheduler_factor"], patience=config["scheduler_patience"], min_lr=config["min_lr"])
    criterion = nn.CrossEntropyLoss()
    
    fold_ckpt_dir = CV_CHECKPOINT_DIR / f"fold_{fold_idx}"
    fold_ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt_path = fold_ckpt_dir / "best.pt"
    
    best_val_loss = float("inf")
    patience_counter = 0
    history = []
    
    for epoch in range(config["max_epochs"]):
        model.train()
        train_loss_total = 0.0
        train_correct = 0
        train_count = 0
        
        pbar = tqdm(train_loader, desc=f"Fold {fold_idx} Epoch {epoch+1}/{config['max_epochs']}", leave=False)
        for images, targets, _ in pbar:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            bs = images.size(0)
            train_loss_total += loss.item() * bs
            preds = outputs.argmax(dim=1)
            train_correct += (preds == targets).sum().item()
            train_count += bs
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{train_correct/train_count:.3f}"})
            
        epoch_train_loss = train_loss_total / train_count
        epoch_train_acc = train_correct / train_count
        
        # Validation
        model.eval()
        val_loss_total = 0.0
        all_val_targets = []
        all_val_preds = []
        with torch.no_grad():
            for images, targets, _ in val_loader:
                images, targets = images.to(DEVICE), targets.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, targets)
                val_loss_total += loss.item() * images.size(0)
                all_val_targets.extend(targets.cpu().numpy().tolist())
                all_val_preds.extend(outputs.argmax(dim=1).cpu().numpy().tolist())
                
        epoch_val_loss = val_loss_total / len(val_ds)
        val_metrics = calculate_macro_metrics(all_val_targets, all_val_preds, NUM_CLASSES)
        scheduler.step(epoch_val_loss)
        current_lr = optimizer.param_groups[0]["lr"]
        
        saved_best = False
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            patience_counter = 0
            saved_best = True
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "val_loss": best_val_loss,
                "val_metrics": val_metrics,
                "fold": fold_idx
            }, best_ckpt_path)
        else:
            patience_counter += 1
            
        rec = {
            "epoch": epoch + 1, "train_loss": epoch_train_loss, "train_acc": epoch_train_acc,
            "val_loss": epoch_val_loss, "val_acc": val_metrics["accuracy"], "val_f1": val_metrics["macro_f1"],
            "lr": current_lr, "saved_best": saved_best
        }
        history.append(rec)
        print(f"  Ep {epoch+1:02d} | Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.3f} | Val Loss: {epoch_val_loss:.4f} Acc: {val_metrics['accuracy']:.3f} F1: {val_metrics['macro_f1']:.3f} | Best: {'YES' if saved_best else 'NO'}")
        
        if patience_counter >= config["early_stopping_patience"]:
            print(f"  [EARLY STOPPING] Fold {fold_idx} berhenti pada epoch {epoch+1}.")
            break
            
    return {"fold": fold_idx, "best_val_loss": best_val_loss, "best_checkpoint": str(best_ckpt_path), "history": history}

if CV_METRICS_PATH.exists():
    with open(CV_METRICS_PATH, "r", encoding="utf-8") as f:
        cv_results = json.load(f)
    print(f"[LOADED] Riwayat 5-Fold CV dimuat dari {CV_METRICS_PATH}.")
elif RUN_CROSS_VALIDATION and len(cv_folds_df) > 0:
    print("[RUNNING] Menjalankan 5-Fold Cross-Validation secara berurutan...")
    cv_results = []
    for f in range(CV_CONFIG["num_folds"]):
        res = run_cv_fold_training(f, cv_folds_df, CV_CONFIG)
        cv_results.append(res)
    with open(CV_METRICS_PATH, "w", encoding="utf-8") as f:
        json.dump(cv_results, f, indent=2)
    print(f"[COMPLETED] 5-Fold CV selesai. Riwayat disimpan ke {CV_METRICS_PATH}")
else:
    print("[STATUS] RUN_CROSS_VALIDATION is False. Eksekusi CV dilewati.")
    cv_results = []

# 22. Cross-Validation Results Summary

Tabel rekapitulasi performa 5-Fold Out-Of-Fold Cross-Validation:
- Val Loss rata-rata dan per fold.
- Val Accuracy rata-rata dan per fold.
- Macro F1 rata-rata across all 5 folds.

In [ ]:
# ==============================================================================
# SECTION 22: CROSS-VALIDATION RESULTS SUMMARY
# ==============================================================================
def summarize_cv_results(results: List[Dict[str, Any]]):
    if not results:
        print("[INFO] Riwayat CV belum tersedia.")
        return
        
    records = []
    for r in results:
        last_or_best = min(r["history"], key=lambda h: h["val_loss"])
        records.append({
            "Fold": r["fold"],
            "Best Epoch": last_or_best["epoch"],
            "Best Val Loss": f"{last_or_best['val_loss']:.4f}",
            "Val Accuracy": f"{last_or_best['val_acc']*100:.2f}%",
            "Val Macro F1": f"{last_or_best['val_f1']*100:.2f}%",
            "Checkpoint": Path(r["best_checkpoint"]).name
        })
        
    df_cv = pd.DataFrame(records)
    print("REKAPITULASI 5-FOLD OUT-OF-FOLD CV TEACHER:")
    print(df_cv.to_string(index=False))
    
    val_losses = [min(r["history"], key=lambda h: h["val_loss"])["val_loss"] for r in results]
    val_accs = [min(r["history"], key=lambda h: h["val_loss"])["val_acc"] for r in results]
    val_f1s = [min(r["history"], key=lambda h: h["val_loss"])["val_f1"] for r in results]
    
    print("-" * 60)
    print(f"RATA-RATA VAL LOSS    : {np.mean(val_losses):.4f} +/- {np.std(val_losses):.4f}")
    print(f"RATA-RATA VAL ACCURACY: {np.mean(val_accs)*100:.2f}% +/- {np.std(val_accs)*100:.2f}%")
    print(f"RATA-RATA MACRO F1    : {np.mean(val_f1s)*100:.2f}% +/- {np.std(val_f1s)*100:.2f}%")

summarize_cv_results(cv_results)

# 23. Final Teacher Training Configuration

Setelah memverifikasi stabilitas melalui 5-Fold CV, kita mengonfigurasi pelatihan **Final Teacher** menggunakan seluruh 40 citra berlabel yang sah:
- Menggunakan seluruh 40 sampel memberikan sinyal pelatihan maksimal kepada Teacher sebelum tahap pseudo-labeling.
- **Catatan Validasi**: Karena seluruh 40 sampel digunakan untuk training, tidak ada holdout validation independen pada tahap ini; jumlah epoch (15 epoch) dipandu oleh epoch optimal dari CV.

In [ ]:
# ==============================================================================
# SECTION 23: FINAL TEACHER TRAINING CONFIGURATION
# ==============================================================================
FINAL_TEACHER_CONFIG = {
    "experiment_id": "fewshot_pseudolabel_b2_teacher_final",
    "total_training_samples": 40,
    "batch_size": 8,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 15,
    "optimizer": "AdamW",
    "loss": "CrossEntropyLoss"
}

print("KONFIGURASI FINAL TEACHER TRAINING:")
for k, v in FINAL_TEACHER_CONFIG.items():
    print(f"  {k:24}: {v}")

# 24. Manual Final Teacher Training (All 40 Samples)

Pelatihan Final Teacher pada 40 citra berlabel:
- Dilindungi oleh flag `RUN_TEACHER_TRAINING`.
- Checkpoint disimpan ke `models/checkpoints/fewshot_pseudolabel_b2_v1/teacher/best.pt`.
- Riwayat epoch disimpan ke `outputs/metrics/fewshot_pseudolabel_b2_teacher.json`.

In [ ]:
# ==============================================================================
# SECTION 24: MANUAL FINAL TEACHER TRAINING
# ==============================================================================
TEACHER_BEST_CKPT = TEACHER_CHECKPOINT_DIR / "best.pt"
TEACHER_METRICS_PATH = OUTPUT_METRICS_DIR / "fewshot_pseudolabel_b2_teacher.json"

def train_final_teacher(labeled_df: pd.DataFrame, config: Dict[str, Any]) -> Dict[str, Any]:
    dataset = FewShotLabeledDataset(labeled_df, PROJECT_ROOT, transform=train_transforms)
    loader = DataLoader(dataset, batch_size=config["batch_size"], shuffle=True, num_workers=0)
    
    model = build_teacher_model(num_classes=NUM_CLASSES, pretrained=True, freeze_backbone=True).to(DEVICE)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    criterion = nn.CrossEntropyLoss()
    
    history = []
    print(f"Melatih Final Teacher pada {len(dataset)} citra selama {config['epochs']} epoch...")
    
    for epoch in range(config["epochs"]):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(loader, desc=f"Final Teacher Ep {epoch+1}/{config['epochs']}", leave=False)
        for images, targets, _ in pbar:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            bs = images.size(0)
            total_loss += loss.item() * bs
            preds = outputs.argmax(dim=1)
            correct += (preds == targets).sum().item()
            total += bs
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{correct/total:.3f}"})
            
        ep_loss = total_loss / total
        ep_acc = correct / total
        history.append({"epoch": epoch + 1, "train_loss": ep_loss, "train_acc": ep_acc})
        print(f"  Ep {epoch+1:02d} | Train Loss: {ep_loss:.4f} | Train Acc: {ep_acc*100:.2f}%")
        
    torch.save({
        "epoch": config["epochs"],
        "model_state_dict": model.state_dict(),
        "final_loss": ep_loss,
        "final_acc": ep_acc,
        "class_names": CLASS_NAMES,
        "config": config
    }, TEACHER_BEST_CKPT)
    
    return {"final_loss": ep_loss, "final_acc": ep_acc, "history": history}

if TEACHER_BEST_CKPT.exists() and TEACHER_METRICS_PATH.exists():
    with open(TEACHER_METRICS_PATH, "r", encoding="utf-8") as f:
        teacher_results = json.load(f)
    print(f"[LOADED] Checkpoint Final Teacher tersedia: {TEACHER_BEST_CKPT}")
elif RUN_TEACHER_TRAINING and len(labeled_pool_df) > 0:
    teacher_results = train_final_teacher(labeled_pool_df, FINAL_TEACHER_CONFIG)
    with open(TEACHER_METRICS_PATH, "w", encoding="utf-8") as f:
        json.dump(teacher_results, f, indent=2)
    print(f"[SAVED] Final Teacher Checkpoint disimpan ke {TEACHER_BEST_CKPT}")
else:
    print("[STATUS] RUN_TEACHER_TRAINING is False. Training Final Teacher dilewati.")
    teacher_results = {}

# 25. Teacher Training History & Curves

Plot kurva pelatihan Final Teacher (loss dan akurasi) dari riwayat persisten pada disk.

In [ ]:
# ==============================================================================
# SECTION 25: TEACHER TRAINING HISTORY & CURVES
# ==============================================================================
def plot_teacher_history(results: Dict[str, Any]):
    if not results or "history" not in results:
        print("[INFO] Riwayat pelatihan Final Teacher belum tersedia.")
        return
        
    epochs = [h["epoch"] for h in results["history"]]
    losses = [h["train_loss"] for h in results["history"]]
    accs = [h["train_acc"] for h in results["history"]]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    ax1.plot(epochs, losses, "o-", color="firebrick", label="Train Loss")
    ax1.set_title("Final Teacher Training Loss", weight="bold")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.grid(True, linestyle="--", alpha=0.6)
    ax1.legend()
    
    ax2.plot(epochs, accs, "s-", color="navy", label="Train Accuracy")
    ax2.set_title("Final Teacher Training Accuracy", weight="bold")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.grid(True, linestyle="--", alpha=0.6)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

plot_teacher_history(teacher_results)

# 26. Teacher Validation Diagnostics

Diskusi saintifik mengenai performa Teacher Model:
- Pada rejim few-shot (5 sampel/kelas), model berbasis transfer learning EfficientNet-B2 mampu menangkap representasi visual umum dari ImageNet.
- Namun, varians estimasi tinggi karena ukuran validasi yang kecil.
- Oleh karena itu, Teacher membutuhkan **threshold confidence yang tinggi (>= 0.95)** pada tahap pseudo-labeling agar tidak mentransfer noise atau misklasifikasi ke model Student.

In [ ]:
# ==============================================================================
# SECTION 26: TEACHER VALIDATION DIAGNOSTICS
# ==============================================================================
print("RINGKASAN DIAGNOSTIK TEACHER MODEL:")
print("1. Backbone     : Pretrained ImageNet-1K mempertahankan representasi filter kaya.")
print("2. Classifier   : 8 kelas sampah terhubung dengan 1408 fitur backbone.")
print("3. Risiko       : Konfirmasi bias (confirmation bias) jika pseudo-label tidak disaring ketat.")
print("4. Rekomendasi  : Gunakan confidence threshold >= 0.95 pada Section 30 untuk menyaring pseudo-label.")

# 27. Unlabeled Inference Setup

Mempersiapkan inferensi model Teacher pada **Unlabeled Pool** (~1.876 citra):
- Memeriksa ketersediaan checkpoint Final Teacher (`TEACHER_BEST_CKPT`).
- Memuat arsitektur `EfficientNet-B2` dengan `weights=None` dan memuat `state_dict` langsung dari disk.
- Mengatur model ke mode `model.eval()` dan menggunakan `torch.inference_mode()` untuk efisiensi komputasi VRAM.

In [ ]:
# ==============================================================================
# SECTION 27: UNLABELED INFERENCE SETUP
# ==============================================================================
def load_trained_teacher_model(ckpt_path: Path, device: torch.device) -> nn.Module:
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint teacher tidak ditemukan: {ckpt_path}. Jalankan Section 24 terlebih dahulu.")
    
    model = build_teacher_model(num_classes=NUM_CLASSES, pretrained=False, freeze_backbone=True)
    payload = torch.load(ckpt_path, map_location=device)
    state_dict = payload["model_state_dict"] if "model_state_dict" in payload else payload
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

print(f"Target Checkpoint Teacher: {TEACHER_BEST_CKPT} (Exists: {TEACHER_BEST_CKPT.exists()})")
if TEACHER_BEST_CKPT.exists():
    inference_teacher = load_trained_teacher_model(TEACHER_BEST_CKPT, DEVICE)
    print("[READY] Model Teacher siap untuk inferensi pada data unlabeled.")
else:
    print("[WAITING] Model Teacher belum dilatih. Selesaikan training Section 24 terlebih dahulu.")
    inference_teacher = None

# 28. Manual Pseudo-Label Generation (Raw Predictions)

Menghasilkan prediksi pseudo-label mentah dari Teacher pada seluruh 1.876 citra unlabeled:
- Dilindungi oleh flag `RUN_PSEUDO_LABEL_GENERATION`.
- Menghitung probabilitas softmax untuk setiap citra.
- Mencatat: `image_id`, `relative_path`, `predicted_class`, `predicted_idx`, `confidence`.
- Menyimpan hasil mentah ke `outputs/metrics/pseudo_labels_raw.csv`.

In [ ]:
# ==============================================================================
# SECTION 28: MANUAL PSEUDO-LABEL GENERATION
# ==============================================================================
RAW_PSEUDO_PATH = OUTPUT_METRICS_DIR / "pseudo_labels_raw.csv"

def generate_raw_pseudo_labels(
    model: nn.Module,
    unlabeled_df: pd.DataFrame,
    batch_size: int = 16,
    device: torch.device = DEVICE
) -> pd.DataFrame:
    dataset = UnlabeledDataset(unlabeled_df, PROJECT_ROOT, transform=eval_transforms)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    records = []
    print(f"Menghasilkan pseudo-label mentah pada {len(dataset)} citra unlabeled...")
    
    model.eval()
    with torch.inference_mode():
        for images, image_ids in tqdm(loader, desc="Pseudo-Labeling Inference"):
            images = images.to(device)
            logits = model(images)
            probs = F.softmax(logits, dim=1)
            confidences, preds = torch.max(probs, dim=1)
            
            for img_id, pred_idx, conf in zip(image_ids, preds.cpu().numpy(), confidences.cpu().numpy()):
                records.append({
                    "image_id": str(img_id),
                    "predicted_idx": int(pred_idx),
                    "predicted_class": IDX_TO_CLASS[int(pred_idx)],
                    "confidence": float(conf)
                })
                
    df_pred = pd.DataFrame(records)
    merged = pd.merge(unlabeled_df[["image_id", "relative_path", "sha256"]], df_pred, on="image_id")
    merged.to_csv(RAW_PSEUDO_PATH, index=False)
    return merged

if RAW_PSEUDO_PATH.exists():
    raw_pseudo_df = pd.read_csv(RAW_PSEUDO_PATH)
    print(f"[LOADED] Raw pseudo-labels dimuat dari {RAW_PSEUDO_PATH} ({len(raw_pseudo_df)} baris).")
elif RUN_PSEUDO_LABEL_GENERATION and inference_teacher is not None and len(unlabeled_pool_df) > 0:
    raw_pseudo_df = generate_raw_pseudo_labels(inference_teacher, unlabeled_pool_df, batch_size=16)
    print(f"[COMPLETED] Raw pseudo-labels berhasil disimpan ke {RAW_PSEUDO_PATH}")
else:
    print("[STATUS] RUN_PSEUDO_LABEL_GENERATION is False atau prasyarat belum terpenuhi.")
    raw_pseudo_df = pd.DataFrame()

# 29. Confidence Score Analysis

Analisis distribusi skor keyakinan (confidence) model Teacher pada data unlabeled:
- Histogram probabilitas [0.0 - 1.0].
- Persentil distribusi confidence (25%, 50%, 75%, 90%, 95%).
- Proporsi sampel yang memenuhi berbagai ambang batas (>= 0.80, >= 0.85, >= 0.90, >= 0.95).

In [ ]:
# ==============================================================================
# SECTION 29: CONFIDENCE SCORE ANALYSIS
# ==============================================================================
def analyze_confidence_scores(df: pd.DataFrame):
    if len(df) == 0:
        print("[INFO] Data raw pseudo-labels belum tersedia untuk dianalisis.")
        return
        
    confs = df["confidence"].values
    print("DISTRIBUSI SKOR CONFIDENCE:")
    print(f"  Rata-rata    : {np.mean(confs):.4f}")
    print(f"  Median (50%) : {np.median(confs):.4f}")
    print(f"  Kuantil 75%  : {np.percentile(confs, 75):.4f}")
    print(f"  Kuantil 90%  : {np.percentile(confs, 90):.4f}")
    print(f"  Kuantil 95%  : {np.percentile(confs, 95):.4f}")
    
    thresholds = [0.80, 0.85, 0.90, 0.95, 0.98]
    print()
    print("PROPORSI SAMPEL PER THRESHOLD:")
    for th in thresholds:
        count = (confs >= th).sum()
        pct = (count / len(confs)) * 100
        print(f"  >= {th:.2f} : {count:4d} citra ({pct:.1f}%)")
        
    plt.figure(figsize=(9, 4.5))
    plt.hist(confs, bins=30, color="teal", edgecolor="black", alpha=0.8)
    plt.axvline(0.95, color="red", linestyle="--", linewidth=2, label="Threshold Rekomendasi (0.95)")
    plt.title("Distribusi Skor Kepercayaan (Confidence) Prediksi Teacher", weight="bold")
    plt.xlabel("Confidence Score")
    plt.ylabel("Frekuensi")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

analyze_confidence_scores(raw_pseudo_df)

# 30. Pseudo-Label Filtering (Threshold >= 0.95)

Penyaringan pseudo-label dengan ambang batas ketat:
- Parameter: `CONFIDENCE_THRESHOLD = 0.95` (dapat disesuaikan jika diperlukan).
- Hanya citra dengan `confidence >= 0.95` yang diterima ke dalam pool pelatihan Student.
- Dilindungi oleh flag `RUN_PSEUDO_LABEL_SELECTION`.
- Manifest disimpan ke `data/manifests/fewshot_pseudolabel_b2_v1/pseudo_labels_accepted.csv`.

In [ ]:
# ==============================================================================
# SECTION 30: PSEUDO-LABEL FILTERING
# ==============================================================================
CONFIDENCE_THRESHOLD = 0.95
ACCEPTED_PSEUDO_PATH = MANIFEST_DIR / "pseudo_labels_accepted.csv"

def filter_pseudo_labels(raw_df: pd.DataFrame, threshold: float = 0.95) -> pd.DataFrame:
    filtered = raw_df[raw_df["confidence"] >= threshold].copy().reset_index(drop=True)
    filtered.to_csv(ACCEPTED_PSEUDO_PATH, index=False)
    return filtered

if ACCEPTED_PSEUDO_PATH.exists():
    accepted_pseudo_df = pd.read_csv(ACCEPTED_PSEUDO_PATH)
    print(f"[LOADED] Accepted pseudo-labels dimuat dari {ACCEPTED_PSEUDO_PATH} ({len(accepted_pseudo_df)} baris).")
elif RUN_PSEUDO_LABEL_SELECTION and len(raw_pseudo_df) > 0:
    accepted_pseudo_df = filter_pseudo_labels(raw_pseudo_df, threshold=CONFIDENCE_THRESHOLD)
    print(f"[CREATED] {len(accepted_pseudo_df)} citra diterima dengan threshold >= {CONFIDENCE_THRESHOLD}.")
else:
    print("[STATUS] RUN_PSEUDO_LABEL_SELECTION is False atau raw pseudo-labels belum tersedia.")
    accepted_pseudo_df = pd.DataFrame()

# 31. Class Balance Analysis (Accepted Pseudo-Labels)

Pemeriksaan keseimbangan kelas dari pseudo-label yang diterima untuk mendeteksi apakah terjadi bias model ekstrem atau dominasi kelas tunggal (class collapse).

In [ ]:
# ==============================================================================
# SECTION 31: CLASS BALANCE ANALYSIS
# ==============================================================================
def plot_pseudo_class_balance(accepted_df: pd.DataFrame):
    if len(accepted_df) == 0:
        print("[INFO] Belum ada accepted pseudo-labels untuk divisualisasikan.")
        return
        
    counts = accepted_df["predicted_class"].value_counts().reindex(CLASS_NAMES).fillna(0)
    plt.figure(figsize=(10, 4.5))
    bars = plt.bar(counts.index, counts.values, color="coral", edgecolor="black")
    plt.title(f"Distribusi Kelas Accepted Pseudo-Labels (Total: {len(accepted_df)} Citra)", weight="bold")
    plt.xlabel("Kelas Prediksi")
    plt.ylabel("Jumlah Citra Diterima")
    plt.xticks(rotation=45)
    for bar in bars:
        h = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., h + 0.5, f"{int(h)}", ha="center", va="bottom", weight="bold")
    plt.tight_layout()
    plt.show()

plot_pseudo_class_balance(accepted_pseudo_df)

# 32. Pseudo-Label Consistency Audit

Audit integritas dan konsistensi manifest pseudo-label:
- Memastikan ID unik (tanpa duplikasi).
- Memastikan seluruh kelas termasuk dalam 8 kelas resmi.
- Memastikan tidak ada tumpang tindih dengan 40 labeled samples atau test set.

In [ ]:
# ==============================================================================
# SECTION 32: PSEUDO-LABEL CONSISTENCY AUDIT
# ==============================================================================
def audit_pseudo_label_manifest(accepted_df: pd.DataFrame, labeled_df: pd.DataFrame, test_dir: Path):
    if len(accepted_df) == 0:
        print("[INFO] Manifest pseudo-labels kosong, audit dilewati.")
        return
        
    assert accepted_df["image_id"].is_unique, "Duplikasi ID ditemukan di accepted pseudo-labels!"
    invalid_classes = set(accepted_df["predicted_class"]) - set(CLASS_NAMES)
    assert len(invalid_classes) == 0, f"Kelas tidak valid: {invalid_classes}"
    overlap_labeled = set(accepted_df["sha256"]) & set(labeled_df["sha256"])
    assert len(overlap_labeled) == 0, f"Bocoran: Pseudo-label tumpang tindih dengan labeled pool! ({len(overlap_labeled)})"
    
    print("[AUDIT VALID] Manifest accepted pseudo-labels konsisten dan bebas anomali.")

audit_pseudo_label_manifest(accepted_pseudo_df, labeled_pool_df, DATA_TEST_DIR)

# 33. Accepted Pseudo-Label Manifest Summary

Ringkasan manifest akhir pseudo-label yang diterima dan siap digabungkan dengan labeled pool untuk melatih model Student.

In [ ]:
# ==============================================================================
# SECTION 33: ACCEPTED PSEUDO-LABEL MANIFEST SUMMARY
# ==============================================================================
if len(accepted_pseudo_df) > 0:
    print(f"Total Accepted Pseudo-Labels : {len(accepted_pseudo_df)} citra")
    total_unlabeled = len(unlabeled_pool_df) if len(unlabeled_pool_df) > 0 else 1876
    print(f"Persentase Diterima          : {len(accepted_pseudo_df)/total_unlabeled*100:.2f}% dari unlabeled pool")
    print()
    print("Preview 5 Baris Pertama:")
    print(accepted_pseudo_df[["image_id", "predicted_class", "confidence", "relative_path"]].head())
else:
    print("[INFO] Accepted pseudo-labels belum dibangkitkan.")

# 34. Student Model Initialization

Inisialisasi model **Student**:
- Arsitektur: `EfficientNet-B2` pretrained ImageNet-1K (`weights=EfficientNet_B2_Weights.IMAGENET1K_V1`).
- Penggantian classifier head menjadi 8 kelas dengan mempertahankan Dropout (0.3).
- Model Student dilatih pada gabungan data: **40 citra berlabel manual** + **citra pseudo-label terpilih**.

In [ ]:
# ==============================================================================
# SECTION 34: STUDENT MODEL INITIALIZATION
# ==============================================================================
def build_student_model(
    num_classes: int = 8,
    pretrained: bool = True,
    freeze_backbone: bool = True
) -> nn.Module:
    weights = EfficientNet_B2_Weights.IMAGENET1K_V1 if pretrained else None
    model = efficientnet_b2(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    
    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False
            
    return model

student_model = build_student_model(num_classes=NUM_CLASSES, pretrained=True, freeze_backbone=True)
student_model = student_model.to(DEVICE)
print("[STUDENT INITIALIZED] Model Student EfficientNet-B2 siap dikonfigurasi.")

# 35. Student Dataset Construction (Combined Semi-Supervised)

Konstruksi kelas Dataset gabungan untuk Student (`CombinedStudentDataset`):
- Memadukan sampel **Manual Supervised** (bobot sampel = 1.0) dan sampel **Accepted Pseudo-Label** (bobot sampel = $\lambda_{pseudo}$).
- Mengembalikan: `(image_tensor, target_class_idx, is_pseudo_flag, sample_weight, image_id)`.

In [ ]:
# ==============================================================================
# SECTION 35: STUDENT DATASET CONSTRUCTION
# ==============================================================================
class CombinedStudentDataset(Dataset):
    def __init__(
        self,
        labeled_df: pd.DataFrame,
        pseudo_df: pd.DataFrame,
        project_root: Path,
        transform: Optional[Callable] = None,
        pseudo_weight: float = 0.5
    ):
        self.project_root = project_root
        self.transform = transform
        self.records = []
        
        # 1. Masukkan sampel berlabel asli
        for _, row in labeled_df.iterrows():
            self.records.append({
                "relative_path": row["relative_path"],
                "target_idx": int(row["class_idx"]),
                "is_pseudo": False,
                "weight": 1.0,
                "image_id": str(row["image_id"])
            })
            
        # 2. Masukkan sampel pseudo-labeled
        for _, row in pseudo_df.iterrows():
            self.records.append({
                "relative_path": row["relative_path"],
                "target_idx": int(row["predicted_idx"]),
                "is_pseudo": True,
                "weight": float(pseudo_weight),
                "image_id": str(row["image_id"])
            })
            
    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int, bool, float, str]:
        item = self.records[idx]
        img_path = self.project_root / item["relative_path"]
        with Image.open(img_path) as img:
            image = img.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
            
        return image, item["target_idx"], item["is_pseudo"], item["weight"], item["image_id"]

print("[DATASET STUDENT TERDEFINISI] CombinedStudentDataset siap memadukan data berlabel dan pseudo-label.")

# 36. Supervised and Pseudo-Labeled Joint Loss

Formulasi fungsi loss semi-supervised terbobot:
$$\mathcal{L}_{total} = rac{1}{N_{sup}} \sum_{i \in sup} 	ext{CE}(y_i, \hat{y}_i) + \lambda_{pseudo} \cdot rac{1}{N_{pseudo}} \sum_{j \in pseudo} 	ext{CE}(	ilde{y}_j, \hat{y}_j)$$
di mana $\lambda_{pseudo} = 0.5$ secara default untuk mencegah noise pseudo-label mendominasi gradient.

In [ ]:
# ==============================================================================
# SECTION 36: SUPERVISED AND PSEUDO-LABELED LOSS
# ==============================================================================
class SemiSupervisedLoss(nn.Module):
    def __init__(self, pseudo_weight: float = 0.5):
        super().__init__()
        self.pseudo_weight = pseudo_weight
        self.ce = nn.CrossEntropyLoss(reduction="none")

    def forward(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
        is_pseudo: torch.Tensor,
        sample_weights: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        losses = self.ce(logits, targets)
        
        sup_mask = ~is_pseudo
        pseudo_mask = is_pseudo
        
        if sup_mask.sum() > 0:
            sup_loss = (losses[sup_mask] * sample_weights[sup_mask]).mean()
        else:
            sup_loss = torch.tensor(0.0, device=logits.device)
            
        if pseudo_mask.sum() > 0:
            pseudo_loss = (losses[pseudo_mask] * sample_weights[pseudo_mask]).mean()
        else:
            pseudo_loss = torch.tensor(0.0, device=logits.device)
            
        total_loss = sup_loss + (self.pseudo_weight * pseudo_loss)
        return total_loss, sup_loss, pseudo_loss

semi_sup_criterion = SemiSupervisedLoss(pseudo_weight=0.5)
print("[LOSS TERDEFINISI] SemiSupervisedLoss dengan bobot pseudo = 0.5 siap digunakan.")

# 37. Student Training Configuration

Konfigurasi pelatihan Student Model:
- Optimizer: `AdamW` (learning rate = 5e-4, weight decay = 1e-4)
- Batch Size: 8
- Maksimum Epoch: 25
- Pseudo Loss Weight: 0.5
- Augmentasi: Transformasi pelatihan terkendali (Section 13)

In [ ]:
# ==============================================================================
# SECTION 37: STUDENT TRAINING CONFIGURATION
# ==============================================================================
STUDENT_CONFIG = {
    "experiment_id": "fewshot_pseudolabel_b2_student",
    "batch_size": 8,
    "learning_rate": 5e-4,
    "weight_decay": 1e-4,
    "epochs": 25,
    "pseudo_loss_weight": 0.5,
    "optimizer": "AdamW"
}

print("KONFIGURASI STUDENT TRAINING:")
for k, v in STUDENT_CONFIG.items():
    print(f"  {k:22}: {v}")

# 38. Manual Student Training Execution

Pelatihan model Student pada dataset semi-supervised gabungan:
- Dilindungi oleh flag `RUN_STUDENT_TRAINING`.
- Memantau running accuracy, supervised loss, dan pseudo loss per batch menggunakan `tqdm.auto`.
- Menyimpan checkpoint Student ke `models/checkpoints/fewshot_pseudolabel_b2_v1/student/best.pt`.
- Menyimpan riwayat metrik ke `outputs/metrics/fewshot_pseudolabel_b2_student.json`.

In [ ]:
# ==============================================================================
# SECTION 38: MANUAL STUDENT TRAINING EXECUTION
# ==============================================================================
STUDENT_BEST_CKPT = STUDENT_CHECKPOINT_DIR / "best.pt"
STUDENT_METRICS_PATH = OUTPUT_METRICS_DIR / "fewshot_pseudolabel_b2_student.json"

def train_student_model(
    labeled_df: pd.DataFrame,
    pseudo_df: pd.DataFrame,
    config: Dict[str, Any]
) -> Dict[str, Any]:
    dataset = CombinedStudentDataset(labeled_df, pseudo_df, PROJECT_ROOT, transform=train_transforms, pseudo_weight=config["pseudo_loss_weight"])
    loader = DataLoader(dataset, batch_size=config["batch_size"], shuffle=True, num_workers=0)
    
    model = build_student_model(num_classes=NUM_CLASSES, pretrained=True, freeze_backbone=True).to(DEVICE)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    criterion = SemiSupervisedLoss(pseudo_weight=config["pseudo_loss_weight"])
    
    print(f"Melatih Model Student pada {len(dataset)} citra gabungan ({len(labeled_df)} manual + {len(pseudo_df)} pseudo)...")
    history = []
    
    for epoch in range(config["epochs"]):
        model.train()
        total_loss_accum = 0.0
        sup_loss_accum = 0.0
        pseudo_loss_accum = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(loader, desc=f"Student Ep {epoch+1}/{config['epochs']}", leave=False)
        for images, targets, is_pseudo, weights, _ in pbar:
            images = images.to(DEVICE)
            targets = targets.to(DEVICE)
            is_pseudo = is_pseudo.to(DEVICE)
            weights = weights.to(DEVICE)
            
            optimizer.zero_grad()
            logits = model(images)
            loss, sup_l, p_l = criterion(logits, targets, is_pseudo, weights)
            loss.backward()
            optimizer.step()
            
            bs = images.size(0)
            total_loss_accum += loss.item() * bs
            sup_loss_accum += sup_l.item() * bs
            pseudo_loss_accum += p_l.item() * bs
            preds = logits.argmax(dim=1)
            correct += (preds == targets).sum().item()
            total += bs
            
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{correct/total:.3f}"})
            
        ep_loss = total_loss_accum / total
        ep_sup_loss = sup_loss_accum / total
        ep_p_loss = pseudo_loss_accum / total
        ep_acc = correct / total
        
        rec = {
            "epoch": epoch + 1, "total_loss": ep_loss, "sup_loss": ep_sup_loss,
            "pseudo_loss": ep_p_loss, "accuracy": ep_acc
        }
        history.append(rec)
        print(f"  Ep {epoch+1:02d} | Loss: {ep_loss:.4f} (Sup: {ep_sup_loss:.4f}, Pseudo: {ep_p_loss:.4f}) | Acc: {ep_acc*100:.2f}%")
        
    torch.save({
        "epoch": config["epochs"],
        "model_state_dict": model.state_dict(),
        "final_loss": ep_loss,
        "final_acc": ep_acc,
        "class_names": CLASS_NAMES,
        "config": config,
        "num_labeled": len(labeled_df),
        "num_pseudo": len(pseudo_df)
    }, STUDENT_BEST_CKPT)
    
    return {"final_loss": ep_loss, "final_acc": ep_acc, "history": history}

if STUDENT_BEST_CKPT.exists() and STUDENT_METRICS_PATH.exists():
    with open(STUDENT_METRICS_PATH, "r", encoding="utf-8") as f:
        student_results = json.load(f)
    print(f"[LOADED] Checkpoint Student tersedia: {STUDENT_BEST_CKPT}")
elif RUN_STUDENT_TRAINING and len(labeled_pool_df) > 0 and len(accepted_pseudo_df) > 0:
    student_results = train_student_model(labeled_pool_df, accepted_pseudo_df, STUDENT_CONFIG)
    with open(STUDENT_METRICS_PATH, "w", encoding="utf-8") as f:
        json.dump(student_results, f, indent=2)
    print(f"[SAVED] Checkpoint Student berhasil disimpan ke {STUDENT_BEST_CKPT}")
else:
    print("[STATUS] RUN_STUDENT_TRAINING is False atau data pseudo belum siap.")
    student_results = {}

# 39. Student Training History Summary

Tabel riwayat metrik pelatihan Student Model (Total Loss, Supervised Loss, Pseudo Loss, dan Running Accuracy) per epoch.

In [ ]:
# ==============================================================================
# SECTION 39: STUDENT TRAINING HISTORY SUMMARY
# ==============================================================================
if student_results and "history" in student_results:
    df_student_hist = pd.DataFrame(student_results["history"])
    print("RIWAYAT PELATIHAN STUDENT MODEL:")
    print(df_student_hist.to_string(index=False))
else:
    print("[INFO] Riwayat pelatihan Student belum tersedia.")

# 40. Student Learning Curves

Visualisasi kurva pembelajaran Student:
- Dekomposisi Total Loss, Supervised Loss, dan Pseudo Loss.
- Kurva akurasi pelatihan.
- Gambar disimpan ke `outputs/figures/fewshot_pseudolabel_b2_student_curves.png`.

In [ ]:
# ==============================================================================
# SECTION 40: STUDENT LEARNING CURVES
# ==============================================================================
STUDENT_CURVES_PATH = OUTPUT_FIGURES_DIR / "fewshot_pseudolabel_b2_student_curves.png"

def plot_student_curves(results: Dict[str, Any]):
    if not results or "history" not in results:
        print("[INFO] Kurva student belum tersedia.")
        return
        
    hist = results["history"]
    epochs = [h["epoch"] for h in hist]
    total_l = [h["total_loss"] for h in hist]
    sup_l = [h["sup_loss"] for h in hist]
    p_l = [h["pseudo_loss"] for h in hist]
    acc = [h["accuracy"] for h in hist]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    ax1.plot(epochs, total_l, "o-", color="purple", label="Total Loss")
    ax1.plot(epochs, sup_l, "--", color="blue", label="Supervised Loss")
    ax1.plot(epochs, p_l, ":", color="orange", label="Pseudo Loss")
    ax1.set_title("Student Semi-Supervised Loss", weight="bold")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True, linestyle="--", alpha=0.5)
    
    ax2.plot(epochs, acc, "s-", color="forestgreen", label="Training Accuracy")
    ax2.set_title("Student Training Accuracy", weight="bold")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(True, linestyle="--", alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(STUDENT_CURVES_PATH, dpi=150)
    plt.show()
    print(f"[FIGURE SAVED] Kurva disimpan ke {STUDENT_CURVES_PATH}")

plot_student_curves(student_results)

# 41. Student Evaluation (Internal Labeled Fit)

Evaluasi kecocokan internal model Student pada 40 sampel manual berlabel:
- Memeriksa apakah penambahan data pseudo-labeled tetap menjaga atau meningkatkan kemampuan klasifikasi pada gold-standard 40 sampel berlabel manual.

In [ ]:
# ==============================================================================
# SECTION 41: STUDENT EVALUATION
# ==============================================================================
def evaluate_student_on_labeled(model_ckpt: Path, labeled_df: pd.DataFrame, device: torch.device):
    if not model_ckpt.exists() or len(labeled_df) == 0:
        print("[INFO] Model Student atau Labeled Pool belum tersedia.")
        return
        
    model = build_student_model(num_classes=NUM_CLASSES, pretrained=False, freeze_backbone=True)
    payload = torch.load(model_ckpt, map_location=device)
    model.load_state_dict(payload["model_state_dict"])
    model.to(device)
    model.eval()
    
    dataset = FewShotLabeledDataset(labeled_df, PROJECT_ROOT, transform=eval_transforms)
    loader = DataLoader(dataset, batch_size=8, shuffle=False)
    
    all_targets = []
    all_preds = []
    with torch.inference_mode():
        for images, targets, _ in loader:
            images = images.to(device)
            logits = model(images)
            preds = logits.argmax(dim=1)
            all_targets.extend(targets.numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            
    eval_met = calculate_macro_metrics(all_targets, all_preds, NUM_CLASSES)
    print("HASIL EVALUASI STUDENT PADA 40 CITRA LABELED POOL:")
    print(f"  Akurasi  : {eval_met['accuracy']*100:.2f}%")
    print(f"  Macro F1 : {eval_met['macro_f1']*100:.2f}%")

if STUDENT_BEST_CKPT.exists():
    evaluate_student_on_labeled(STUDENT_BEST_CKPT, labeled_pool_df, DEVICE)

# 42. Teacher vs Student Comparison

Analisis komparatif menyeluruh antara model Teacher dan model Student:
- **Teacher**: Dilatih hanya pada 40 citra berlabel manual.
- **Student**: Dilatih pada 40 citra berlabel manual + citra pseudo-label yang lolos filter (threshold >= 0.95).
- **Integritas Metrik**: Perbandingan disajikan secara jujur. Tidak boleh membandingkan metrik out-of-fold Teacher dengan metrik in-sample Student sebagai klaim peningkatan semu.

In [ ]:
# ==============================================================================
# SECTION 42: TEACHER VS STUDENT COMPARISON
# ==============================================================================
def compare_teacher_and_student():
    records = []
    
    # Data Teacher
    if TEACHER_BEST_CKPT.exists():
        t_payload = torch.load(TEACHER_BEST_CKPT, map_location="cpu")
        records.append({
            "Model": "Final Teacher",
            "Backbone": "EfficientNet-B2",
            "Supervised Samples": 40,
            "Pseudo Samples": 0,
            "Final Loss": f"{t_payload.get('final_loss', float('nan')):.4f}",
            "Training Acc": f"{t_payload.get('final_acc', float('nan'))*100:.2f}%",
            "Role": "Baseline & Pseudo-Label Generator"
        })
        
    # Data Student
    if STUDENT_BEST_CKPT.exists():
        s_payload = torch.load(STUDENT_BEST_CKPT, map_location="cpu")
        records.append({
            "Model": "Student Semi-Sup",
            "Backbone": "EfficientNet-B2",
            "Supervised Samples": 40,
            "Pseudo Samples": s_payload.get("num_pseudo", 0),
            "Final Loss": f"{s_payload.get('final_loss', float('nan')):.4f}",
            "Training Acc": f"{s_payload.get('final_acc', float('nan'))*100:.2f}%",
            "Role": "Final Classifier with Distillation"
        })
        
    if records:
        df_comp = pd.DataFrame(records)
        print("TABEL PERBANDINGAN TEACHER VS STUDENT:")
        print(df_comp.to_string(index=False))
    else:
        print("[INFO] Model Teacher atau Student belum selesai dilatih.")

if RUN_TEACHER_STUDENT_EVALUATION or (TEACHER_BEST_CKPT.exists() and STUDENT_BEST_CKPT.exists()):
    compare_teacher_and_student()
else:
    print("[STATUS] RUN_TEACHER_STUDENT_EVALUATION is False. Evaluasi komparatif dilewati.")

# 43. Validation Metrics

Rekapitulasi metrik validasi resmi dari Out-Of-Fold Cross-Validation:
- Akurasi
- Macro Precision
- Macro Recall
- Macro F1-Score across all 8 classes.

In [ ]:
# ==============================================================================
# SECTION 43: VALIDATION METRICS
# ==============================================================================
if CV_METRICS_PATH.exists():
    with open(CV_METRICS_PATH, "r", encoding="utf-8") as f:
        cv_data = json.load(f)
    print("METRIK VALIDASI OUT-OF-FOLD (5-FOLD CV):")
    for r in cv_data:
        best_ep = min(r["history"], key=lambda h: h["val_loss"])
        print(f"  Fold {r['fold']}: Loss={best_ep['val_loss']:.4f} | Acc={best_ep['val_acc']*100:.2f}% | F1={best_ep['val_f1']*100:.2f}%")
else:
    print("[INFO] Metrik CV validasi belum tersedia.")

# 44. Classification Report

Laporan klasifikasi per kelas untuk 8 kelas sampah target (Precision, Recall, F1-Score, dan Support) menggunakan `sklearn.metrics.classification_report`.

In [ ]:
# ==============================================================================
# SECTION 44: CLASSIFICATION REPORT
# ==============================================================================
def display_classification_report(y_true: List[int], y_pred: List[int]):
    print("CLASSIFICATION REPORT (8 KELAS TARGET):")
    rep = metrics.classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4, zero_division=0)
    print(rep)

# Contoh evaluasi pada labeled pool jika checkpoint teacher tersedia
if TEACHER_BEST_CKPT.exists() and len(labeled_pool_df) > 0:
    t_model = load_trained_teacher_model(TEACHER_BEST_CKPT, DEVICE)
    ds = FewShotLabeledDataset(labeled_pool_df, PROJECT_ROOT, transform=eval_transforms)
    ld = DataLoader(ds, batch_size=8, shuffle=False)
    y_t, y_p = [], []
    with torch.inference_mode():
        for imgs, tgts, _ in ld:
            logits = t_model(imgs.to(DEVICE))
            y_t.extend(tgts.numpy().tolist())
            y_p.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    display_classification_report(y_t, y_p)
else:
    print("[INFO] Checkpoint Teacher belum tersedia untuk classification report.")

# 45. Confusion Matrix (Raw & Normalized)

Visualisasi Confusion Matrix untuk menganalisis kesalahan prediksi antarkelas (misalnya antara `cardboard` vs `paper` atau `glass` vs `plastic`).

In [ ]:
# ==============================================================================
# SECTION 45: CONFUSION MATRIX
# ==============================================================================
def plot_confusion_matrices(y_true: List[int], y_pred: List[int]):
    cm_raw = metrics.confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    cm_norm = cm_raw.astype("float") / (cm_raw.sum(axis=1)[:, np.newaxis] + 1e-9)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
    
    im1 = ax1.imshow(cm_raw, interpolation="nearest", cmap=plt.cm.Blues)
    ax1.set_title("Confusion Matrix (Hitungan Mentah)", weight="bold")
    fig.colorbar(im1, ax=ax1)
    tick_marks = np.arange(NUM_CLASSES)
    ax1.set_xticks(tick_marks)
    ax1.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax1.set_yticks(tick_marks)
    ax1.set_yticklabels(CLASS_NAMES)
    ax1.set_xlabel("Prediksi")
    ax1.set_ylabel("True Label")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax1.text(j, i, format(cm_raw[i, j], "d"), ha="center", va="center", color="white" if cm_raw[i, j] > cm_raw.max()/2 else "black")
            
    im2 = ax2.imshow(cm_norm, interpolation="nearest", cmap=plt.cm.Greens)
    ax2.set_title("Confusion Matrix (Ternormalisasi)", weight="bold")
    fig.colorbar(im2, ax=ax2)
    ax2.set_xticks(tick_marks)
    ax2.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax2.set_yticks(tick_marks)
    ax2.set_yticklabels(CLASS_NAMES)
    ax2.set_xlabel("Prediksi")
    ax2.set_ylabel("True Label")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax2.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", color="white" if cm_norm[i, j] > 0.5 else "black")
            
    plt.tight_layout()
    plt.show()

if 'y_t' in locals() and len(y_t) > 0:
    plot_confusion_matrices(y_t, y_p)
else:
    print("[INFO] Data evaluasi belum tersedia.")

# 46. Overfitting / Underfitting Analysis

Analisis stabilitas model pada skenario data sedikit (Few-Shot):
1. **Risiko Overfitting**: Dengan hanya 5 citra per kelas, model rentan menghafal latar belakang (background memorization).
2. **Mitigasi**: Pembekuan backbone (frozen backbone), Dropout 0.3, Weight Decay 1e-4, dan augmentasi fotorealistik.
3. **Risiko Pseudo-Labeling**: Confirmation bias terjadi jika threshold terlalu rendah (< 0.80). Threshold 0.95 meminimalkan transfer label salah ke model Student.

In [ ]:
# ==============================================================================
# SECTION 46: OVERFITTING / UNDERFITTING ANALYSIS
# ==============================================================================
print("ANALISIS RESIKO DAN REGULARISASI:")
print("1. Parameter Ratio : 11.272 parameter trainable pada 40 citra (rasio ~281 param/citra).")
print("2. Freeze Backbone : 7,70 juta parameter backbone dibekukan untuk menjaga stabilitas.")
print("3. Pseudo Filter   : Threshold 0.95 membatasi hanya prediksi berkeyakinan tinggi yang masuk ke Student.")
print("4. Loss Weighting  : Bobot pseudo 0.5 memberikan prioritas lebih tinggi pada label asli berbobot 1.0.")

# 47. Experiment Summary

Ringkasan eksekutif eksperimen TASK 08:
- Status kuota label (40/40 terpakai).
- Status Teacher dan Student model.
- Checkpoint yang siap digunakan untuk tahapan inferensi submission.

In [ ]:
# ==============================================================================
# SECTION 47: EXPERIMENT SUMMARY
# ==============================================================================
def print_experiment_summary():
    print("=" * 70)
    print("RINGKASAN EKSEKUTIF EKSPERIMEN TASK 08:")
    print(f"  Project Root       : {PROJECT_ROOT.name}")
    print(f"  Backbone Model     : EfficientNet-B2 (Pretrained ImageNet-1K)")
    print(f"  Batas Kuota Label  : 40 Citra Manual (5 per kelas x 8 kelas) - DIPATUHI")
    print(f"  Unlabeled Pool     : {len(unlabeled_pool_df)} citra")
    print(f"  Accepted Pseudo    : {len(accepted_pseudo_df)} citra (Threshold >= {CONFIDENCE_THRESHOLD})")
    print(f"  Teacher Checkpoint : {'Tersedia' if TEACHER_BEST_CKPT.exists() else 'Belum Dilatih'}")
    print(f"  Student Checkpoint : {'Tersedia' if STUDENT_BEST_CKPT.exists() else 'Belum Dilatih'}")
    print("=" * 70)

print_experiment_summary()

# 48. Competition Test Audit (480 Citra)

Audit menyeluruh terhadap direktori data test kompetisi (`data/competition/test/`) dan template `SampleSubmission_DSC.csv`:
- Tepat **480 citra test** berekstensi `.jpg`.
- Pola penamaan ID: `001.jpg` sampai `480.jpg` (mempertahankan leading zero).
- Sample submission memiliki tepat 480 baris data dengan header `image_id,waste_category`.
- **Anti-Leakage**: Nilai kategori pada sample submission adalah dummy dan dilarang digunakan sebagai target.

In [ ]:
# ==============================================================================
# SECTION 48: COMPETITION TEST AUDIT
# ==============================================================================
def audit_competition_test_data(test_dir: Path, sample_sub_path: Path):
    if not test_dir.exists():
        raise FileNotFoundError(f"Direktori test tidak ditemukan: {test_dir}")
    if not sample_sub_path.exists():
        raise FileNotFoundError(f"Sample submission tidak ditemukan: {sample_sub_path}")
        
    test_files = sorted(list(test_dir.glob("*.jpg")), key=lambda p: p.name)
    sample_df = pd.read_csv(sample_sub_path, dtype={"image_id": str})
    
    print("HASIL AUDIT DATA TEST KOMPETISI:")
    print(f"  Jumlah Citra Test   : {len(test_files)} (Diharapkan 480)")
    print(f"  Jumlah Baris Sample : {len(sample_df)} (Diharapkan 480)")
    print(f"  Kolom Sample        : {list(sample_df.columns)}")
    print(f"  Contoh ID Citra     : {[p.stem for p in test_files[:5]]}")
    
    assert len(test_files) == 480, f"Jumlah citra test ({len(test_files)}) tidak sesuai dengan 480!"
    assert len(sample_df) == 480, f"Jumlah baris sample submission ({len(sample_df)}) tidak sesuai dengan 480!"
    print("[TEST AUDIT PASS] Data test kompetisi terverifikasi utuh dan valid.")

audit_competition_test_data(DATA_TEST_DIR, SAMPLE_SUBMISSION_PATH)

# 49. Standalone Inference Configuration (Fresh-Kernel Compatible)

**KRUSIAL**: Bagian ini didesain agar dapat dijalankan langsung dari **kernel Jupyter yang baru/bersih** tanpa perlu menjalankan cell training sebelumnya:
1. Mengimpor dependensi secara mandiri.
2. Mendeteksi `PROJECT_ROOT` secara dinamis.
3. Mendefinisikan arsitektur `EfficientNet-B2` (`weights=None`) dan memuat `state_dict` langsung dari checkpoint disk.
4. Menyiapkan transform inferensi standar (288x288, ImageNet normalization).
5. Membaca template `SampleSubmission_DSC.csv` untuk menjaga urutan dan format ID resmi.

In [ ]:
# ==============================================================================
# SECTION 49: STANDALONE INFERENCE CONFIGURATION
# ==============================================================================
# Standalone imports
import sys
from pathlib import Path
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b2
from torchvision.transforms import v2

def standalone_setup():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    root = Path.cwd()
    for c in candidates:
        if (c / "data" / "competition").exists() or (c / "data" / "extracted").exists():
            root = c.resolve()
            break
            
    classes = ["cardboard", "electronic", "foodwaste", "glass", "metal", "paper", "plastic", "textile"]
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    transform = v2.Compose([
        v2.ToImage(),
        v2.Resize(288, interpolation=v2.InterpolationMode.BICUBIC, antialias=True),
        v2.CenterCrop(288),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return root, classes, dev, transform

SA_ROOT, SA_CLASSES, SA_DEVICE, SA_TRANSFORM = standalone_setup()
print(f"[STANDALONE READY] Root: {SA_ROOT} | Device: {SA_DEVICE} | Classes: {len(SA_CLASSES)}")

# 50. Best Model Selection for Test Inference

Logika pemilihan checkpoint terbaik untuk inferensi test kompetisi:
- **Prioritas 1**: Model **Student** (`student/best.pt`) jika telah dilatih menggunakan semi-supervised learning.
- **Prioritas 2**: Model **Teacher** (`teacher/best.pt`) jika Student belum dilatih.
- Jika keduanya belum tersedia, fungsi akan memberikan pesan informatif bahwa training perlu dijalankan terlebih dahulu.

In [ ]:
# ==============================================================================
# SECTION 50: BEST MODEL SELECTION FOR TEST INFERENCE
# ==============================================================================
def select_best_inference_checkpoint(project_root: Path) -> Tuple[Path, str]:
    student_ckpt = project_root / "models" / "checkpoints" / "fewshot_pseudolabel_b2_v1" / "student" / "best.pt"
    teacher_ckpt = project_root / "models" / "checkpoints" / "fewshot_pseudolabel_b2_v1" / "teacher" / "best.pt"
    
    if student_ckpt.exists():
        return student_ckpt, "Student (Few-Shot + Pseudo-Labeling)"
    elif teacher_ckpt.exists():
        return teacher_ckpt, "Teacher (Few-Shot Supervised Only)"
    else:
        raise FileNotFoundError(
            f"Tidak ditemukan checkpoint model di {student_ckpt.parent} atau {teacher_ckpt.parent}. "
            "Jalankan Section 24 (Teacher) atau Section 38 (Student) terlebih dahulu!"
        )

try:
    SELECTED_CKPT_PATH, SELECTED_MODEL_TYPE = select_best_inference_checkpoint(PROJECT_ROOT)
    print(f"CHECKPOINT TERPILIH : {SELECTED_CKPT_PATH}")
    print(f"TIPE MODEL          : {SELECTED_MODEL_TYPE}")
except FileNotFoundError as err:
    print(f"[WAITING] {err}")
    SELECTED_CKPT_PATH, SELECTED_MODEL_TYPE = None, None

# 51. Manual Competition Test Inference (480 Citra)

Menjalankan inferensi pada seluruh 480 citra test kompetisi:
- Dilindungi oleh flag `RUN_TEST_INFERENCE`.
- Menggunakan `model.eval()` dan `torch.inference_mode()`.
- Menghasilkan prediksi kelas resmi untuk setiap `image_id`.

In [ ]:
# ==============================================================================
# SECTION 51: MANUAL COMPETITION TEST INFERENCE
# ==============================================================================
def predict_competition_test(
    ckpt_path: Path,
    test_dir: Path,
    sample_sub_path: Path,
    device: torch.device
) -> Dict[str, str]:
    sample_df = pd.read_csv(sample_sub_path, dtype={"image_id": str})
    expected_ids = sample_df["image_id"].tolist()
    
    model = build_teacher_model(num_classes=NUM_CLASSES, pretrained=False, freeze_backbone=True)
    payload = torch.load(ckpt_path, map_location=device)
    state_dict = payload["model_state_dict"] if "model_state_dict" in payload else payload
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    
    test_files = [test_dir / f"{img_id}.jpg" for img_id in expected_ids]
    test_ds = CompetitionTestDataset(test_files, transform=eval_transforms)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=0)
    
    predictions = {}
    print(f"Menjalankan inferensi pada {len(test_ds)} citra test...")
    with torch.inference_mode():
        for images, img_ids in tqdm(test_loader, desc="Test Inference"):
            images = images.to(device)
            logits = model(images)
            preds = logits.argmax(dim=1).cpu().numpy()
            for img_id, pred_idx in zip(img_ids, preds):
                predictions[str(img_id)] = IDX_TO_CLASS[int(pred_idx)]
                
    return predictions

if RUN_TEST_INFERENCE and SELECTED_CKPT_PATH is not None:
    test_predictions = predict_competition_test(SELECTED_CKPT_PATH, DATA_TEST_DIR, SAMPLE_SUBMISSION_PATH, DEVICE)
    print(f"[COMPLETED] Inferensi selesai. {len(test_predictions)} prediksi berhasil dibuat.")
else:
    print("[STATUS] RUN_TEST_INFERENCE is False atau checkpoint belum terpilih.")
    test_predictions = {}

# 52. Submission CSV Generation

Pembuatan berkas CSV submission resmi:
- Target File: `Final Dataset for Submission to Kaggle/submission_fewshot_pseudolabel_b2.csv`.
- Format Kolom: `image_id,waste_category`.
- Dilindungi oleh flag `RUN_SUBMISSION_GENERATION`.
- Menjaga leading zeros pada `image_id` (`001`, `002`, dst.) dan menyelaraskan urutan baris dengan `SampleSubmission_DSC.csv`.

In [ ]:
# ==============================================================================
# SECTION 52: SUBMISSION CSV GENERATION
# ==============================================================================
def generate_submission_csv(
    predictions: Dict[str, str],
    sample_sub_path: Path,
    output_path: Path
) -> pd.DataFrame:
    sample_df = pd.read_csv(sample_sub_path, dtype={"image_id": str})
    
    records = []
    for img_id in sample_df["image_id"]:
        if img_id not in predictions:
            raise KeyError(f"Prediksi untuk image_id '{img_id}' tidak ditemukan!")
        records.append({
            "image_id": str(img_id),
            "waste_category": str(predictions[img_id])
        })
        
    sub_df = pd.DataFrame(records)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sub_df.to_csv(output_path, index=False)
    return sub_df

if RUN_SUBMISSION_GENERATION and len(test_predictions) == 480:
    submission_df = generate_submission_csv(test_predictions, SAMPLE_SUBMISSION_PATH, SUBMISSION_CSV_PATH)
    print(f"[SUCCESS] Submission CSV disimpan ke: {SUBMISSION_CSV_PATH}")
else:
    print("[STATUS] RUN_SUBMISSION_GENERATION is False atau prediksi test belum lengkap (480).")
    submission_df = pd.DataFrame()

# 53. Submission CSV Verification (Standalone Checklist)

Cell verifikasi mandiri terhadap berkas CSV submission yang dihasilkan:
1. File fisik benar-benar ada di disk.
2. Tepat **480 baris data** + 1 baris header.
3. Nama kolom persis: `image_id,waste_category`.
4. Leading zero pada `image_id` (`001`, `002`, dst.) terjaga utuh.
5. Tidak ada nilai kosong (NaN/null/whitespace).
6. Seluruh kategori termasuk dalam 8 kelas resmi kompetisi.
7. Urutan `image_id` identik dengan `SampleSubmission_DSC.csv`.

In [ ]:
# ==============================================================================
# SECTION 53: SUBMISSION CSV VERIFICATION
# ==============================================================================
def verify_submission_csv(csv_path: Path, sample_sub_path: Path):
    if not csv_path.exists():
        print(f"[INFO] Berkas submission belum ada di {csv_path}. Jalankan Section 52 terlebih dahulu.")
        return False
        
    sub_df = pd.read_csv(csv_path, dtype={"image_id": str})
    sample_df = pd.read_csv(sample_sub_path, dtype={"image_id": str})
    
    assert len(sub_df) == 480, f"Jumlah baris ({len(sub_df)}) tidak tepat 480!"
    assert list(sub_df.columns) == ["image_id", "waste_category"], f"Kolom ({list(sub_df.columns)}) tidak sesuai!"
    assert sub_df.isnull().sum().sum() == 0, "Ditemukan nilai NaN/null pada submission!"
    invalid = set(sub_df["waste_category"]) - set(CLASS_NAMES)
    assert len(invalid) == 0, f"Ditemukan kelas tidak valid: {invalid}"
    assert (sub_df["image_id"] == sample_df["image_id"]).all(), "Urutan image_id tidak selaras dengan sample submission!"
    first_id = str(sub_df.iloc[0]["image_id"])
    assert len(first_id) == 3 and first_id.startswith("0"), f"Format leading zero rusak: {first_id}"
    
    print("=" * 60)
    print("HASIL VERIFIKASI SUBMISSION CSV:")
    print(f"  File Path      : {csv_path.name}")
    print(f"  Jumlah Baris   : {len(sub_df)} baris (VALID)")
    print(f"  Format Kolom   : {list(sub_df.columns)} (VALID)")
    print(f"  Leading Zero   : Terjaga (Contoh: {first_id}) (VALID)")
    print(f"  ID Alignment   : 100% Cocok dengan Sample Submission (VALID)")
    print("=" * 60)
    print()
    print("Preview 5 Baris Pertama:")
    print(sub_df.head())
    print()
    print("Distribusi Prediksi Tiap Kelas:")
    print(sub_df["waste_category"].value_counts())
    return True

verify_submission_csv(SUBMISSION_CSV_PATH, SAMPLE_SUBMISSION_PATH)

# 54. Final Submission Summary & Kaggle Upload Guide

Rangkuman akhir dan panduan pengunggahan hasil ke platform kompetisi Kaggle:
- File Submission: `Final Dataset for Submission to Kaggle/submission_fewshot_pseudolabel_b2.csv`.
- Sesuai regulasi kompetisi, **pengunggahan ke Kaggle dilakukan secara manual oleh pengguna**.
- Pipeline telah mematuhi seluruh batasan teknis: maksimal 40 label manual (5 per kelas), mitigasi kebocoran data, dan penyaringan pseudo-label berkualitas tinggi.

In [ ]:
# ==============================================================================
# SECTION 54: FINAL SUBMISSION SUMMARY & KAGGLE UPLOAD GUIDE
# ==============================================================================
def print_final_submission_guide():
    print("=" * 75)
    print("PANDUAN AKHIR PENGUNGGAHAN SUBMISSION KAGGLE:")
    print("1. Berkas submission resmi siap di:")
    print(f"   {SUBMISSION_CSV_PATH}")
    print("2. Buka halaman kompetisi di platform Kaggle.")
    print("3. Unggah file CSV tersebut ke menu 'Submit Predictions'.")
    print("4. Berikan deskripsi model:")
    print(f"   'Few-Shot EfficientNet-B2 (40 manual labels) + Pseudo-Labeling (th>=0.95)'")
    print("5. Periksa skor Public Leaderboard setelah proses evaluasi Kaggle selesai.")
    print("=" * 75)

print_final_submission_guide()